# Qiskit Fall Fest 2026, Session 2: Quantum algorithms

Tue, Nov 17, 2026, 5:00 to 6:00 PM, on Zoom

Covers: Grover's search, the quantum Fourier transform, QAOA, and VQE.

This is the second of five sessions this week. Tonight has three parts.

- Seminar: four algorithms, one pattern.
- Lab: Grover on two qubits, then QAOA for Max-Cut.
- Challenge 2 opens at 6:00 PM. Challenge 1 stays open until Sunday.

The words in the "Covers" line, in one sentence each. An algorithm is a step-by-step recipe for solving a problem, and a quantum algorithm is one whose steps run on qubits, the quantum version of bits. Grover's search finds a marked item in an unsorted list in far fewer steps than checking the items one by one. The quantum Fourier transform, QFT for short, is a building block inside larger algorithms. QAOA, the quantum approximate optimization algorithm, looks for good answers to optimization problems, where the task is to find the best choice among very many. Tonight's optimization problem is called Max-Cut: split a network into two groups so that as many connections as possible run between the groups. VQE, the variational quantum eigensolver, estimates the lowest energy that a system such as a molecule can have. Qiskit is the open-source Python library, developed by IBM, that you use all week to write quantum programs and run them.

This notebook stands on its own. If you missed Session 1, start here anyway: Part 1 opens with the few ideas from Monday that tonight needs, and every other term is explained where it first appears.

## What you will be able to do

- Explain a quantum algorithm as interference arranged on purpose, and run Grover's search on two qubits. Interference is what happens when amplitudes, the numbers that describe a quantum state, add up or cancel.
- Say what the quantum Fourier transform does to the amplitudes of a state.
- Turn a Max-Cut problem into a cost operator and a QAOA circuit, and count the gates that transpiling adds. Max-Cut asks you to split a network into two groups so that as many connections as possible run between the groups. A cost operator is the scoring rule of a problem, written in a form that a quantum computer can use. A circuit is a list of operations on qubits, the operations are called gates, and transpiling means rewriting a circuit for one particular quantum computer.
- Score sampled bitstrings, the strings of 0s and 1s that measurements return, and pick the best cut.
- Run a small VQE and compare its energy with the exact value.

## How to use this notebook

- Run the cells in order, from the top. A cell is one box of text or code. To run a code cell, click it and press Shift+Enter.
- Part 1 follows tonight's slides. Each idea has a short recap, and often a small cell you can run to see the idea at work.
- Part 2 is the lab. The code under "Try it" and "Lab" is complete. Run it, read the output, then change it and run it again.
- After each lab there is a "Your turn" exercise. Replace the line `# YOUR CODE HERE` with your own code, run the cell, then run the check cell below it. A check that fails prints a hint.
- Everything runs on a simulator, a program that imitates a quantum computer, on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser), unless you change the switches in the Setup section.
- Use this notebook during the session and again afterwards on your own.

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## Join the Discord and register

*Slide 2.*

- **Discord.** Join the server at https://discord.gg/FBNzRcxMr. Questions between sessions go there.
- **Registration.** Not registered yet? Fill in the form at https://forms.gle/NC9o1iRk3WggLuTH7.

On the slide the two links are QR codes, small square patterns that a phone camera reads as a link. Discord is the chat service where the club answers questions.

### New tonight?

*Slide 4.*

- **IBM Quantum Platform.** quantum.cloud.ibm.com. Open Plan instance and an API key. This is where your circuits run.
- **qBraid, the second option.** account.qbraid.com. Sign up with the email you registered with. Access is sent there.
- **Step by step.** The setup slides are in the Session 1 deck on the event page.

Already set up? Open tonight's notebook and run the first cell.

You need an account only to run on a real quantum computer. Everything in this notebook runs without one, on the simulator, so you can do the whole lab tonight and set up an account later.

Some words from the first card. IBM Quantum Platform is the website where you manage your IBM Quantum account. An API key is a secret string of characters that lets a program, such as this notebook, act for your account. A plan is the kind of access an account has, and the Open Plan is the one you do not pay for. An instance is your share of access under a plan. Each instance has a CRN, short for cloud resource name: the long identifier that IBM Quantum Platform lists with your instances. With `USE_HARDWARE = True`, the first hardware cell asks for it, together with the API key, when no account is saved yet.

The Session 1 notebook has the same steps as cells you can run, in its "Accounts" section: IBM Quantum Platform first, then the second option. A job is one request to run circuits, and a processor is the chip that holds the qubits. Jobs on IBM processors always go through your own IBM Quantum account. The second option does not replace that account. It adds the other quantum computers that qBraid offers.

In this notebook "the first cell" means the three code cells of the Setup section above. If they ran without an error, you are ready.

## Part 1: The seminar ideas

This part follows tonight's slides in order. Each heading is the title of a slide. Under it you find a short recap, the math from the slide, and often a small cell to run. The sections called "Derivation" show the steps behind a result, one at a time.

**If you missed Session 1.** Tonight builds on a few ideas from Monday. Here they are, each in a sentence or two.

- **Qubit.** The quantum version of a bit. When you measure a qubit, you read a 0 or a 1.
- **Bitstring.** What you read when you measure several qubits: a string of 0s and 1s with one character per qubit, such as `01`. Qiskit numbers the qubits from 0 and prints qubit 0 as the right-most character.
- **Amplitude.** Before the measurement, the state of $n$ qubits is described by one number for each of the $2^n$ possible bitstrings. These numbers are the amplitudes. An amplitude can be negative. In general it is a complex number, a number of the form $x + iy$ with $i^2 = -1$.
- **Probability.** The probability of reading a bitstring is the squared size of its amplitude. The size of $x + iy$ is $\sqrt{x^2 + y^2}$, and the size of an ordinary number is the number without its sign. So the amplitudes $\frac{1}{2}$ and $-\frac{1}{2}$ both give the probability $\frac{1}{4}$.
- **Ket and superposition.** $\lvert 01 \rangle$ is called a ket. It is the state in which a measurement is certain to give 01. A general state is a sum of kets, each multiplied by its amplitude, for example $\frac{1}{\sqrt{2}}\lvert 00 \rangle + \frac{1}{\sqrt{2}}\lvert 01 \rangle$. A state with more than one amplitude different from zero is a superposition.
- **Phase.** The sign of an amplitude or, for a complex amplitude, its direction in the plane of complex numbers. A phase by itself changes no probability.
- **Gate.** An operation that changes the amplitudes without measuring anything. Tonight's gates: H, the Hadamard gate, turns $\lvert 0 \rangle$ into $\frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}$ and $\lvert 1 \rangle$ into $\frac{\lvert 0 \rangle - \lvert 1 \rangle}{\sqrt{2}}$. X flips a qubit between 0 and 1. Z puts a minus sign on every amplitude in which its qubit is 1. CZ acts on two qubits and puts a minus sign on every amplitude in which both are 1.
- **Circuit.** A list of gates applied to qubits, usually ending with a measurement of every qubit.
- **Shots and counts.** One run of a circuit is called a shot, and it gives one bitstring. The counts say how many shots gave each bitstring.

The cell below shows these ideas on a circuit with two qubits. It applies three gates to qubit 0, one after the other: H, Z, H. After each gate it prints the four amplitudes with their labels.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


def amplitudes(circuit):
    """Return the amplitude of every bitstring at the end of a circuit, with the labels Qiskit prints.

    Only the real part is kept. That is the whole amplitude for circuits made of H, X, Z and CZ gates.
    """
    width = circuit.num_qubits
    values = Statevector(circuit).data          # the exact amplitudes, as a list of 2**width numbers
    return {format(k, f"0{width}b"): round(float(values[k].real), 3) + 0.0 for k in range(2 ** width)}


warm_up = QuantumCircuit(2)        # a circuit with two qubits, both starting in 0
print("start:             ", amplitudes(warm_up))
warm_up.h(0)                       # H on qubit 0
print("after H on qubit 0:", amplitudes(warm_up))
warm_up.z(0)                       # Z on qubit 0
print("after Z on qubit 0:", amplitudes(warm_up))
warm_up.h(0)                       # H on qubit 0 again
print("after a second H:  ", amplitudes(warm_up))

Read the output line by line.

- At the start the only amplitude is on `00`: both qubits are 0.
- After H there are two equal amplitudes, on `00` and `01`. The character that changed is the right one, because qubit 0 is printed on the right. Each of the two results now has the probability $0.707^2 \approx 0.5$. The exact amplitude is $\frac{1}{\sqrt{2}}$, and its square is exactly one half.
- After Z the amplitude of `01` has a minus sign. The probabilities are the same as before, so nothing you could measure at this point has changed.
- After the second H the whole amplitude is on `01`. H sends each of the two parts of the state to both `00` and `01`. The two contributions to `00` are $+0.5$ and $-0.5$, and they cancel. The two contributions to `01` are $+0.5$ and $+0.5$, and they add up to 1.

Amplitudes that add up or cancel are said to interfere. The minus sign that no measurement could see decided the result one gate later. Every algorithm tonight is built on this effect.

### A quantum algorithm is interference, arranged on purpose

*Slide 5.*

- Spread amplitude over every answer.
- Mark the good answers with a phase.
- Interfere so the good ones add and the rest cancel.

The three bullets are a recipe, and Grover's search, the first algorithm of tonight, follows it step by step. First, a layer of H gates gives every possible answer, which means every bitstring, the same amplitude. Second, a part of the circuit that depends on the problem puts a phase, for example a minus sign, on the answers you are looking for. That changes no probability yet. Third, more gates make the amplitudes interfere, so that the amplitudes of the marked answers grow and the others shrink. Only then do you measure.

The speedup never comes from trying everything at once. A measurement returns one bitstring, and if you measured right after the first step you would get a random one. The work is done by the interference step, which moves probability toward the good answers before you look.

The picture on the slide is four bar charts: equal amplitudes, one amplitude flipped negative, one amplitude large, and the measured answer. The cell below draws them from a real circuit, the smallest version of Grover's search. It looks for one marked result, 11, among the four results of two qubits. The slide's picture marks 10 instead. The steps are the same for any marked result, and from slide 9 on the marked result is 11. The slides that follow explain every gate of it. The line that starts with `measured =` uses a Sampler, the Qiskit tool that runs a circuit for a number of shots and returns the measured bitstrings. Lab 1 explains that line.

In [ ]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector

search = QuantumCircuit(2)
search.h([0, 1])                             # 1. spread: equal amplitudes on all four results
spread = Statevector(search).data.real       # the four amplitudes, in the order 00, 01, 10, 11
search.cz(0, 1)                              # 2. mark: a minus sign on the amplitude of 11
marked = Statevector(search).data.real
search.h([0, 1])                             # 3. interfere: these five lines are explained further down
search.x([0, 1])
search.cz(0, 1)
search.x([0, 1])
search.h([0, 1])
interfered = Statevector(search).data.real
search.measure_all()                         # 4. measure, with 1,024 shots
measured = StatevectorSampler(seed=2026).run([search], shots=1024).result()[0].data.meas.get_counts()
print("counts:", measured)

labels = ["00", "01", "10", "11"]
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for ax, title, stage in zip(axes, ["1. spread", "2. mark", "3. interfere"], [spread, marked, interfered]):
    ax.bar(labels, stage, color="tab:blue")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylim(-1.1, 1.1)
    ax.set_title(title)
axes[0].set_ylabel("amplitude")
axes[3].bar(labels, [measured.get(label, 0) for label in labels], color="tab:orange")
axes[3].set_title("4. measure")
axes[3].set_ylabel("shots out of 1,024")
fig.tight_layout()
plt.show()

Panel 1: four equal amplitudes of 0.5, so each result has the probability $0.5^2 = 0.25$. Panel 2: the amplitude of 11 has changed its sign, and the probabilities are still 0.25 each. Panel 3: after the interference step only 11 has an amplitude. Panel 4: all 1,024 shots gave 11.

In panel 3 the bar points down: the amplitude of 11 is $-1$, not $+1$. That makes no difference to a measurement, because the probability is the squared size, $(-1)^2 = 1$. A minus sign in front of the whole state is called a global phase, and no measurement can see it. The section "Two qubits, amplitude by amplitude" below shows where this sign comes from.

### Grover finds one item in $N$ with about $\sqrt{N}$ looks

*Slide 6.*

$$k \approx \frac{\pi}{4} \sqrt{\frac{N}{M}}$$

$N$ items, $M$ of them marked. A classical search needs about $N/2$ looks when $M = 1$.

The setting: you have $N$ items in no particular order, and $M$ of them are the ones you want. All you can do is hand an item to a checker that answers "yes, this is one" or "no". Each use of the checker is one look. A classical search, which is a search on an ordinary computer, can only try the items one after another. With one marked item it needs about $N/2$ looks on average.

Grover's search needs about $k$ steps, where $k$ is the number in the formula, and each step uses the checker once. For a million items with one marked, that is $\frac{\pi}{4} \times 1000 \approx 785$ steps, against 500,000 classical looks on average. The number of steps grows like $\sqrt{N}$, not like $N$. This is called a quadratic speedup.

In a circuit the items are the bitstrings: $n$ qubits give $N = 2^n$ of them. One Grover step is two operations in a row, called the oracle and the diffuser. The oracle is the checker: it marks the wanted bitstrings with a minus sign. The diffuser is the interference step. The next sections derive the formula and then show both operations as gates.

#### Derivation: The geometry behind the square root

*Slide 7.*

Collect the marked items into one state, $\lvert w \rangle$, and all the others into a second state, $\lvert r \rangle$, for "rest". $\lvert w \rangle$ is the equal superposition of the $M$ marked bitstrings, and $\lvert r \rangle$ is the equal superposition of the $N - M$ unmarked ones. The whole search takes place in combinations of these two states.

Start state, with $w$ marked and $r$ the rest. The search starts in $\lvert s \rangle$, the state with the same amplitude $\frac{1}{\sqrt{N}}$ on every one of the $N$ bitstrings. $M$ of them are marked, so the probability of reading a marked one is $\frac{M}{N}$. Call that probability $\sin^2\theta$. Then the amplitude of $\lvert w \rangle$ is $\sin\theta$, and the amplitude of $\lvert r \rangle$ is $\cos\theta$, because the two probabilities add up to one and $\sin^2\theta + \cos^2\theta = 1$.

$$\lvert s \rangle = \sin\theta\,\lvert w \rangle + \cos\theta\,\lvert r \rangle, \quad \sin\theta = \sqrt{\frac{M}{N}}$$

After $k$ steps. $G$ stands for one Grover step and $G^k$ for $k$ steps in a row. Each step turns the state by the angle $2\theta$ toward $\lvert w \rangle$. The next section shows why. The start is at the angle $\theta$, so after $k$ steps the angle is $\theta + 2k\theta = (2k+1)\theta$.

$$G^k\lvert s \rangle = \sin((2k+1)\theta)\,\lvert w \rangle + \cos((2k+1)\theta)\,\lvert r \rangle$$

Success probability. The probability of reading a marked item is the squared amplitude of $\lvert w \rangle$.

$$P_k = \sin^2((2k+1)\theta)$$

Stop when the angle reaches $90^\circ$. At $90^\circ$, which is $\frac{\pi}{2}$ when the angle is measured in radians, the unit in which a full turn is $2\pi$, the sine is 1 and the state is exactly $\lvert w \rangle$.

$$(2k+1)\theta = \frac{\pi}{2} \; \Rightarrow \; k \approx \frac{\pi}{4}\sqrt{\frac{N}{M}}$$

The arrow hides two small steps. Solving the equation on the left for $k$ gives the exact value.

$$k = \frac{\pi}{4\theta} - \frac{1}{2}$$

When the marked items are a small part of the list, $\theta$ is a small angle. For a small angle measured in radians, $\sin\theta \approx \theta$, so $\theta \approx \sqrt{\frac{M}{N}}$. This is called the small-angle approximation. Then $\frac{\pi}{4\theta}$ is a large number, the $\frac{1}{2}$ hardly matters, and

$$k \approx \frac{\pi}{4\theta} \approx \frac{\pi}{4}\sqrt{\frac{N}{M}}$$

The number of steps has to be a whole number, so in practice you take the whole number nearest to the exact value. Stopping at the right moment matters: if you run past the best $k$, the angle moves beyond $90^\circ$ and the probability falls again.

### Each Grover step rotates the state by $2\theta$

*Slide 8.*

- Two reflections make a rotation.
- The oracle reflects about the unmarked states.
- The diffuser reflects about the starting state.

Draw the state as an arrow of length 1 in a plane. The horizontal axis is $\lvert r \rangle$, the rest. The vertical axis is $\lvert w \rangle$, the marked items. The start state $\lvert s \rangle$ is the arrow at the angle $\theta$ above the horizontal axis.

The oracle puts a minus sign on the marked part and leaves the rest alone. In the picture that flips the vertical part of the arrow. This is a reflection, a mirror image, about the horizontal axis: an arrow at the angle $\theta$ goes to the angle $-\theta$.

The diffuser reflects about the start state $\lvert s \rangle$. An arrow at the angle $-\theta$ lies $2\theta$ below $\lvert s \rangle$, so its mirror image lies $2\theta$ above $\lvert s \rangle$, at the angle $3\theta$.

Two reflections in a row always make a rotation, by twice the angle between the two mirror lines. Here the mirror lines are $\theta$ apart, so one Grover step rotates the arrow by $2\theta$ toward the marked axis. After $k$ steps the angle is $(2k+1)\theta$. That is the second line of the derivation above.

The cell below draws this picture for $N = 16$ and $M = 1$, where $\sin\theta = \frac{1}{4}$ and $\theta \approx 14.5^\circ$. The dashed arrow is the state after the oracle of the first step.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

theta = np.arcsin(1 / 4)                    # N = 16, M = 1: sin(theta) = sqrt(1/16) = 1/4

fig, ax = plt.subplots(figsize=(4.6, 4.2))
arc = np.linspace(-0.35, 2.4, 200)
ax.plot(np.cos(arc), np.sin(arc), color="0.85")                 # every state is an arrow of length 1
ax.axhline(0, color="0.5", lw=0.8)
ax.axvline(0, color="0.5", lw=0.8)


def arrow(angle, color, style="-"):
    """Draw an arrow of length 1 from the origin, at the given angle."""
    ax.annotate("", xy=(np.cos(angle), np.sin(angle)), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, lw=2, linestyle=style))


arrow(-theta, "0.5", "--")                                      # after the oracle of the first step
ax.text(np.cos(-theta) + 0.03, np.sin(-theta) - 0.03, "after the oracle", color="0.4", va="top", ha="right")
for k, color in zip(range(4), ["black", "tab:blue", "tab:green", "tab:red"]):
    angle = (2 * k + 1) * theta                                 # the angle after k steps
    arrow(angle, color)
    label = "start" if k == 0 else f"after {k} step" + ("s" if k > 1 else "")
    ax.text(1.08 * np.cos(angle), 1.08 * np.sin(angle), label, color=color,
            ha="left" if np.cos(angle) > 0.3 else "center")
    print(label + ":", "angle", round(float(np.degrees(angle)), 1), "degrees | probability of the marked item",
          round(float(np.sin(angle) ** 2), 4))
ax.set_xlim(-0.6, 1.45)
ax.set_ylim(-0.45, 1.25)
ax.set_aspect("equal")
ax.set_xlabel("rest")
ax.set_ylabel("marked item")
ax.set_title("Grover steps for N = 16")
plt.show()

Each step turns the arrow by $2\theta \approx 29^\circ$. After two steps the arrow stands at $72.4^\circ$, short of the vertical axis. After three steps it stands at $101.3^\circ$, a little past it. The probability of the marked item is the square of the arrow's height: 0.908 after two steps and 0.961 after three.

So how many steps for $N = 16$? The exact value from the derivation is $k = \frac{\pi}{4\theta} - \frac{1}{2} \approx 2.6$, and the formula on the slide gives $\frac{\pi}{4}\sqrt{16} \approx 3.14$. The two numbers differ, because $\theta$ is not a very small angle here, but both round to three steps.

The next cell checks the formula against a circuit. It builds Grover's search on four qubits, which is $N = 16$ items with one of them marked, runs it with $k = 1$ to 5 steps, and computes the probability of the marked item each time. `Statevector` computes the exact state at the end of a circuit. Nothing is sampled, so there is no randomness.

Both the oracle and the diffuser use the function `flip_sign`, which puts a minus sign on the amplitude of one chosen bitstring. It works in three moves. X gates turn the chosen bitstring into 1111. A CZ gate on four qubits puts a minus sign on 1111 only, the one bitstring in which all four qubits are 1. In Qiskit it is `mcp` with the angle $\pi$, short for multi-controlled phase. Qubits 0, 1 and 2 are its controls: the gate acts on qubit 3 only when all three of them are 1. The same X gates then undo the first move.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

wanted = "1011"                             # the marked item: one of the 16 results of four qubits


def flip_sign(circuit, bits):
    """Put a minus sign on the amplitude of one bitstring and leave the other 15 alone."""
    zeros = [q for q, bit in enumerate(reversed(bits)) if bit == "0"]    # qubit 0 is the last character
    for q in zeros:
        circuit.x(q)                        # after these X gates, `bits` has become 1111
    circuit.mcp(np.pi, [0, 1, 2], 3)        # a CZ with three controls: a minus sign on 1111 only
    for q in zeros:
        circuit.x(q)                        # undo the X gates


theta = np.arcsin(1 / 4)                    # sin(theta) = sqrt(M / N) = sqrt(1 / 16)
simulated = []
for k in range(1, 6):
    grover16 = QuantumCircuit(4)
    grover16.h(range(4))                    # equal amplitudes on all 16 results
    for step in range(k):
        flip_sign(grover16, wanted)         # the oracle: mark the wanted item
        grover16.h(range(4))                # the diffuser: H on every qubit,
        flip_sign(grover16, "0000")         #   a minus sign on 0000,
        grover16.h(range(4))                #   and H on every qubit again
    p_wanted = float(Statevector(grover16).probabilities_dict()[wanted])
    simulated.append(p_wanted)
    print("k =", k, "| simulated", round(p_wanted, 4), "| formula", round(float(np.sin((2 * k + 1) * theta) ** 2), 4))

smooth = np.linspace(0, 5, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(smooth, np.sin((2 * smooth + 1) * theta) ** 2, color="0.5", label="formula")
ax.plot(range(1, 6), simulated, "o", color="tab:blue", label="simulated circuit")
ax.axvline(np.pi / (4 * theta) - 0.5, color="tab:orange", ls="--", label="angle of 90 degrees")
ax.set_xlabel("number of Grover steps k")
ax.set_ylabel("probability of the marked item")
ax.set_title("Grover's search among N = 16 items")
ax.legend(loc="lower center")
plt.show()

The simulated values agree with the formula to four decimals: 0.4727, 0.9084, 0.9613, 0.5817 and 0.1255 for 1 to 5 steps. Three steps are best. A fourth step lowers the success probability to 0.58 and a fifth to 0.13: more work for a worse result. This happens because every step is a rotation, and a rotation does not stop at the target.

### On two qubits, one step is enough

*Slide 9.*

- $N = 4$ and $M = 1$, so $\sin\theta = 1/2$ and $\theta = 30^\circ$.
- After one step the angle is $3\theta = 90^\circ$.
- Probability of the marked answer: $\sin^2(90^\circ) = 1$.

Two qubits have four possible results: 00, 01, 10 and 11. Mark one of them and the numbers work out exactly. The start angle is $30^\circ$, one step adds $60^\circ$, and the arrow lands on the marked axis. A single look finds the marked item with certainty, where a classical search among four items can need three.

This is the circuit of Lab 1. On a simulator, a program that imitates a quantum computer, the certainty is exact. On a real processor it will be a little less, because gates and measurements go wrong now and then. Friday's session explains why.

The picture on the slide is this circuit, drawn by Qiskit. The cell below builds and draws it. Read a circuit drawing from left to right. Each horizontal line is a qubit. A box is a gate on one qubit, and two dots joined by a line are a CZ gate. The gray bars with a dashed line are barriers. A barrier does nothing to the qubits. Here it only separates the three parts of the drawing.

In [ ]:
from qiskit import QuantumCircuit

grover2 = QuantumCircuit(2)
grover2.h([0, 1])          # equal amplitudes on 00, 01, 10 and 11
grover2.barrier()
grover2.cz(0, 1)           # the oracle: a minus sign on 11
grover2.barrier()
grover2.h([0, 1])          # the diffuser: these five lines
grover2.x([0, 1])
grover2.cz(0, 1)
grover2.x([0, 1])
grover2.h([0, 1])
grover2.draw("mpl")

The three parts, from left to right:

- **H on both qubits.** Equal amplitudes on all four results.
- **The oracle.** One CZ gate, which puts a minus sign on the amplitude of 11. The oracle is the only part of the circuit that knows which result is marked.
- **The diffuser.** H, X, CZ, X, H. The X gates on both sides of the second CZ move its minus sign from 11 to 00. The H gates on both sides turn "a minus sign on 00" into "a minus sign on the starting state $\lvert s \rangle$", because H on every qubit turns $\lvert s \rangle$ into $\lvert 00 \rangle$ and back. Up to an overall sign, that is the reflection about $\lvert s \rangle$ from the slide before. The derivation below follows the four amplitudes through it.

#### Derivation: Two qubits, amplitude by amplitude

*Slide 10.*

Superpose. H on both qubits gives each of the four results the amplitude $\frac{1}{2}$.

$$\lvert s \rangle = \frac{1}{2}\left( \lvert 00 \rangle + \lvert 01 \rangle + \lvert 10 \rangle + \lvert 11 \rangle \right)$$

The oracle flips the marked sign. The CZ gate puts a minus sign on $\lvert 11 \rangle$.

$$\frac{1}{2}\left( \lvert 00 \rangle + \lvert 01 \rangle + \lvert 10 \rangle - \lvert 11 \rangle \right)$$

The diffuser reflects about the mean $m$. Here $a$ stands for any one of the four amplitudes and $m$ for the mean, the average, of all four. The reflection about $\lvert s \rangle$ keeps the mean as it is and reverses the distance of every amplitude from the mean: $a = m + (a - m)$ becomes $m - (a - m) = 2m - a$.

$$a \rightarrow 2m - a, \quad m = \frac{1}{4}\left( \frac{1}{2} + \frac{1}{2} + \frac{1}{2} - \frac{1}{2} \right) = \frac{1}{4}$$

Result. The three unmarked amplitudes are $\frac{1}{2}$. They lie $\frac{1}{4}$ above the mean and land $\frac{1}{4}$ below it, on 0. The marked amplitude is $-\frac{1}{2}$. It lies $\frac{3}{4}$ below the mean and lands $\frac{3}{4}$ above it, on 1.

$$2 \times \frac{1}{4} - \frac{1}{2} = 0, \quad 2 \times \frac{1}{4} + \frac{1}{2} = 1 \; \Rightarrow \; \lvert 11 \rangle$$

This move is called inversion about the mean, and it is the whole trick. The oracle's minus sign pulled the mean down from $\frac{1}{2}$ to $\frac{1}{4}$. Reflecting about that lowered mean sends the three unmarked amplitudes to zero and the marked one to one.

The cell below repeats the derivation with numbers and compares it with the circuit of Lab 1.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

step_by_step = QuantumCircuit(2)
step_by_step.h([0, 1])
superposed = Statevector(step_by_step).data.real        # the four amplitudes, for 00, 01, 10, 11
step_by_step.cz(0, 1)                                   # the oracle
after_oracle = Statevector(step_by_step).data.real
step_by_step.h([0, 1])                                  # the diffuser of Lab 1
step_by_step.x([0, 1])
step_by_step.cz(0, 1)
step_by_step.x([0, 1])
step_by_step.h([0, 1])
after_diffuser = Statevector(step_by_step).data.real

mean = after_oracle.mean()
print("order of the amplitudes:   00, 01, 10, 11")
print("after the H gates:        ", np.round(superposed, 3) + 0.0)
print("after the oracle:         ", np.round(after_oracle, 3) + 0.0)
print("mean m of these four:     ", round(float(mean), 3))
print("2m - a, as on the slide:  ", np.round(2 * mean - after_oracle, 3) + 0.0)
print("after the lab's diffuser: ", np.round(after_diffuser, 3) + 0.0)
print("probabilities at the end: ", np.round(after_diffuser ** 2, 3) + 0.0)

The first lines repeat the derivation: three amplitudes of 0.5 and one of $-0.5$ have the mean 0.25, and $2m - a$ turns them into 0, 0, 0 and 1.

The circuit of Lab 1 ends with 0, 0, 0 and $-1$. Every amplitude has the opposite sign of the slide's result. The reason is the way the diffuser is built. Its middle part, X, CZ, X, puts a minus sign on 00 and leaves the other three results alone. The reflection on the slide needs the opposite in that place: 00 left alone, and a minus sign on the other three. The two differ by a minus sign on everything, which is a global phase. So the circuit ends in $-\lvert 11 \rangle$ where the slide has $\lvert 11 \rangle$, the probabilities are the same, and a measurement gives 11 every time.

### The QFT is a Fourier transform on amplitudes

*Slide 11.*

$$\lvert j \rangle \rightarrow \frac{1}{\sqrt{N}} \sum_{k = 0}^{N - 1} e^{2 \pi i j k / N} \lvert k \rangle$$

Here $N = 2^n$. The circuit is $n$ Hadamards and $n(n-1)/2$ controlled phases.
It is the engine inside phase estimation and Shor's algorithm.

How to read the formula. Number the $N = 2^n$ bitstrings of $n$ qubits from 0 to $N - 1$ by reading each bitstring as a binary number. With two qubits, 00, 01, 10 and 11 are the numbers 0, 1, 2 and 3. The bitstrings are written as Qiskit prints them, so qubit 0 is the last binary digit, the one that counts ones. $\lvert j \rangle$ is the state in which the qubits hold the number $j$. The symbol $\sum_{k=0}^{N-1}$ means: add up the terms for $k = 0, 1, \ldots, N - 1$.

The QFT, the quantum Fourier transform, turns $\lvert j \rangle$ into a superposition of all $N$ numbers $k$. Every amplitude has the same size, $\frac{1}{\sqrt{N}}$, so a measurement would give every number with the same probability. What the input $j$ changes is the phases. The factor $e^{2\pi i jk/N}$ is a complex number of size 1. Its direction in the plane of complex numbers turns by the fraction $\frac{j}{N}$ of a full turn each time $k$ grows by one. So the input number $j$ comes out as a speed of rotation, a frequency. That is what a Fourier transform does in general: it changes between "which position" and "how fast something repeats".

A state that is a superposition of several $\lvert j \rangle$ is transformed term by term, each term with its own amplitude. The result is the discrete Fourier transform of the list of amplitudes, the operation that signal processing applies to lists of numbers, up to the sign convention in the exponent. Hence the title of the slide.

Why this is interesting: the circuit is small. It needs $n$ Hadamard gates and $\frac{n(n-1)}{2}$ controlled-phase gates. A controlled-phase gate acts on two qubits and multiplies the amplitude by a phase $e^{i\varphi}$ when both qubits are 1. CZ is the controlled phase with $\varphi = \pi$, because $e^{i\pi} = -1$. The classical fast Fourier transform, FFT for short, needs on the order of $n\,2^n$ operations for the same $2^n$ numbers. For 30 qubits that is 465 gates against about 32 billion operations.

The catch: you cannot read the amplitudes out. A measurement after the QFT returns one number $k$, not the list of $N$ amplitudes. So the QFT does not replace the classical FFT. It helps only as a step inside a larger algorithm that arranges for the measured number to carry the answer. Phase estimation, which reads out the phase that a gate puts on a state, and Shor's algorithm, which finds the factors of a large number, are the best-known examples. Wednesday's session shows what Shor's algorithm does to encryption.

One detail about the gate count on the slide: it leaves out the SWAP gates at the end of the circuit. A SWAP gate exchanges the states of two qubits, and the QFT ends with $n/2$ of them, rounded down, which reverse the order of the qubits. The next section shows one.

#### Derivation: The QFT on one and two qubits

*Slide 12.*

One qubit: the QFT is the H gate. With $n = 1$ there are $N = 2$ numbers, 0 and 1, and the sum has two terms. The term for $k = 0$ has the phase $e^{0} = 1$, and the term for $k = 1$ has the phase $e^{2\pi i j/2} = e^{i\pi j}$.

$$\lvert j \rangle \rightarrow \frac{1}{\sqrt{2}}\left( \lvert 0 \rangle + e^{i\pi j}\lvert 1 \rangle \right)$$

For $j = 0$ the phase is 1 and the result is $\frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}$. For $j = 1$ the phase is $e^{i\pi} = -1$ and the result is $\frac{\lvert 0 \rangle - \lvert 1 \rangle}{\sqrt{2}}$. That is what H does to $\lvert 0 \rangle$ and $\lvert 1 \rangle$.

Two qubits, input $j = 1$. Now $N = 4$ and the phase of the ket $\lvert k \rangle$ is $e^{2\pi i k/4} = i^k$, because $e^{2\pi i/4}$ is a quarter turn, which is the number $i$. The four phases are $i^0 = 1$, $i^1 = i$, $i^2 = -1$ and $i^3 = -i$.

$$\lvert 1 \rangle \rightarrow \frac{1}{2}\left( \lvert 0 \rangle + i\lvert 1 \rangle - \lvert 2 \rangle - i\lvert 3 \rangle \right)$$

In this line the kets are numbered 0 to 3, and the phase steps around the circle of radius 1 by a quarter turn from each ket to the next. Every amplitude has the size $\frac{1}{2}$.

Gate count. $n$ Hadamard gates plus $\frac{n(n-1)}{2}$ controlled phases:

$$n + \frac{n(n-1)}{2} = \frac{n(n+1)}{2}$$

For two qubits that is 3 gates, and for 30 qubits it is 465. The SWAP gates at the end are not counted in this formula.

Classical FFT on the same data. The data is a list of $2^n$ numbers, and the FFT needs about $n$ operations for each of them. The notation $O(\ldots)$ means "grows in proportion to".

$$O(n\,2^n)$$

The cell below builds the two-qubit QFT from four gates and checks both lines of the derivation. `Operator` turns a circuit into its table of numbers, its matrix. `np.allclose` compares two arrays and ignores rounding errors in the last decimals.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

qft2 = QuantumCircuit(2)
qft2.h(1)
qft2.cp(np.pi / 2, 0, 1)      # controlled phase with the angle pi/2: multiplies the amplitude of 11 by i
qft2.h(0)
qft2.swap(0, 1)               # exchange the two qubits

# The formula of the slide as a table of 4 by 4 numbers: e^(2 pi i j k / 4) / 2 for input j and output k.
formula = np.array([[np.exp(2j * np.pi * j * k / 4) / 2 for j in range(4)] for k in range(4)])
print("the circuit equals the formula:", np.allclose(Operator(qft2).data, formula))

for j in range(4):
    start = Statevector.from_int(j, dims=4)             # the input: two qubits that hold the number j
    after = start.evolve(qft2)                          # apply the circuit
    print("input", j, "-> amplitudes of 0, 1, 2, 3, times 2:", np.round(2 * after.data, 3) + 0.0)

no_swap = QuantumCircuit(2)                             # the same circuit without the last gate
no_swap.h(1)
no_swap.cp(np.pi / 2, 0, 1)
no_swap.h(0)
print("without the swap, input 1 gives:      ", np.round(2 * Statevector.from_int(1, dims=4).evolve(no_swap).data, 3) + 0.0)

The first line says that the four-gate circuit is exactly the formula. Python writes $i$ as `1j`, so `0.+1.j` is $i$ and `0.-1.j` is $-i$. The line for input 1 is the result of the derivation: 1, $i$, $-1$, $-i$, each divided by 2. Input 0 gives four equal amplitudes. Input 2 gives amplitudes that turn by half a turn per step, so the signs alternate. Input 3 turns by three quarters of a turn per step.

About the count: the circuit has two H gates and one controlled phase, the three gates of the formula, plus one SWAP that the formula leaves out. The last line shows what the SWAP is for. Without it the right amplitudes come out on the wrong kets: $i$ and $-1$ have changed places.

One more cell, to show how the QFT earns its place inside a larger algorithm. Take four qubits, so $N = 16$, and prepare a state whose amplitudes repeat: equal amplitudes on every second number, or on every fourth, or on every eighth. The cell applies the QFT to each of these states, using Qiskit's ready-made `QFTGate`, and lists the numbers that a measurement could return afterwards.

In [ ]:
import numpy as np
from qiskit.circuit.library import QFTGate
from qiskit.quantum_info import Statevector

for period in (2, 4, 8):
    repeating = np.zeros(16)
    repeating[1::period] = 1.0                           # equal amplitudes on 1, 1 + period, 1 + 2 * period, ...
    repeating = repeating / np.linalg.norm(repeating)    # rescale so that the probabilities add up to 1
    after_qft = Statevector(repeating).evolve(QFTGate(4))
    possible = [k for k in range(16) if after_qft.probabilities()[k] > 1e-9]
    print("amplitudes repeat every", period, "numbers -> after the QFT a measurement can give:", possible)

An input that repeats every $r$ numbers gives only multiples of $\frac{16}{r}$ after the QFT. So a measurement after the QFT tells you something about how often the pattern repeats, which a measurement of the input itself could not. Change `1::period` to `0::period`, which moves the pattern one place to the left, and the three lists stay the same.

This is the kind of job the QFT does inside phase estimation and Shor's algorithm. Those algorithms arrange for their answer to appear as a repeating pattern in the amplitudes, and the QFT turns the pattern into a number that a measurement can return.

### Max-Cut: split a graph so the most edges cross

*Slide 13.*

$$C(z) = \sum_{(i,j) \in E} \frac{1 - z_i z_j}{2}$$

- Each node gets $z_i = +1$ or $-1$, its side of the cut.
- An edge adds 1 when its ends differ, 0 when they match.
- 12 nodes means $2^{12} = 4{,}096$ possible splits.

A graph is a set of points, called nodes, and a set of lines that each join two nodes, called edges. Think of the nodes as people and of an edge as "these two know each other". A cut splits the nodes into two groups, the two sides. An edge crosses the cut when its two ends are on different sides. Max-Cut is the problem of finding the split with the most crossing edges.

The formula counts the crossing edges. $E$ is the set of edges, and the sum runs over every edge, with $i$ and $j$ its two ends. Every node $i$ gets a number $z_i$, $+1$ for one side and $-1$ for the other, and $z$ stands for the whole list of these numbers. For an edge with both ends on the same side, $z_i z_j = +1$ and the term is $\frac{1 - 1}{2} = 0$. For an edge with its ends on opposite sides, $z_i z_j = -1$ and the term is $\frac{1 + 1}{2} = 1$. So $C(z)$ is the number of edges that cross. It is called the cut value of the split $z$. A function like this one, which gives every possible answer a score, is called a cost function.

With $n$ nodes there are $2^n$ ways to choose the sides, 4,096 of them for 12 nodes. Exchanging the two sides changes no edge, so every cut appears twice in that list, and there are 2,048 different cuts. Tonight's lab graph has 12 nodes. A laptop tries all 4,096 splits in a few thousandths of a second, which is how the lab checks its answers. But every extra node doubles the number of splits, and no fast method is known for the general problem. Max-Cut is what computer scientists call NP-hard: a fast method for it would give a fast method for a large family of hard problems, and most computer scientists expect that no such method exists. That is why approximate methods such as QAOA are of interest.

The picture on the slide is a six-node graph with the nodes on the two sides of the best cut in purple and gray. The cell below draws such a graph. It finds the best cut by brute force, which means trying every possible split.

In [ ]:
import matplotlib.pyplot as plt

small_edges = [(0, 1), (1, 2), (3, 4), (4, 5), (0, 3), (1, 4), (2, 5), (0, 4)]   # six nodes, eight edges
place = {0: (0, 1), 1: (1, 1), 2: (2, 1), 3: (0, 0), 4: (1, 0), 5: (2, 0)}       # where each node is drawn


def edges_cut(sides, edge_list):
    """Count the edges whose two ends are on different sides. sides[i] is 0 or 1 for node i."""
    return sum(sides[i] != sides[j] for i, j in edge_list)


# Brute force: try all 2**6 = 64 ways to put each of the six nodes on side 0 or side 1.
best_sides, best_count = None, -1
for k in range(2 ** 6):
    sides = [int(digit) for digit in format(k, "06b")]      # the number k written as six binary digits
    if edges_cut(sides, small_edges) > best_count:
        best_sides, best_count = sides, edges_cut(sides, small_edges)
print("best split found:", best_sides, "| edges that cross:", best_count, "of", len(small_edges))

fig, ax = plt.subplots(figsize=(4.5, 2.6))
for i, j in small_edges:
    crosses = best_sides[i] != best_sides[j]
    ax.plot([place[i][0], place[j][0]], [place[i][1], place[j][1]], lw=2, zorder=1,
            color="black" if crosses else "tab:red", ls="-" if crosses else ":")
for node, (x, y) in place.items():
    ax.scatter(x, y, s=500, zorder=2, color="tab:purple" if best_sides[node] == 1 else "0.55")
    ax.text(x, y, str(node), ha="center", va="center", color="white", zorder=3)
ax.set_xlim(-0.4, 2.4)
ax.set_ylim(-0.4, 1.4)
ax.axis("off")
ax.set_title("Best cut: " + str(best_count) + " of " + str(len(small_edges)) + " edges cross")
plt.show()

Seven of the eight edges cross between the purple side and the gray side. The dotted red edge is the only one whose two ends have the same color. No split cuts all eight edges, and the triangle in the next section shows why: the dotted edge closes a triangle with nodes 0, 1 and 4.

#### Derivation: Check the cost function on a triangle

*Slide 14.*

Three nodes, three edges. A triangle has the nodes 1, 2 and 3 and an edge between every pair of them, so the sum has three terms.

$$C(z) = \frac{1 - z_1 z_2}{2} + \frac{1 - z_2 z_3}{2} + \frac{1 - z_1 z_3}{2}$$

Everyone on one side. All three products are $+1$, so every term is 0 and no edge crosses.

$$z = (+1, +1, +1): \quad C = 0 + 0 + 0 = 0$$

One node split off. Put node 3 on the other side. The edge between nodes 1 and 2 still has both ends on one side and adds 0. The two edges that end at node 3 cross and add 1 each.

$$z = (+1, +1, -1): \quad C = 0 + 1 + 1 = 2$$

Best possible on a triangle. There are two sides and three nodes, so two of the nodes must share a side, and the edge between those two does not cross.

$$C_{\text{max}} = 2 \text{ of } 3 \text{ edges}$$

A triangle can never have all three edges cut. This is a first look at why Max-Cut is hard: in a larger graph the edges make demands that cannot all be met, and the best split is a compromise.

The cell below evaluates the formula for all $2^3 = 8$ ways to choose the sides.

In [ ]:
print(" z1  z2  z3   cut")
for z1 in (+1, -1):
    for z2 in (+1, -1):
        for z3 in (+1, -1):
            cut = (1 - z1 * z2) / 2 + (1 - z2 * z3) / 2 + (1 - z1 * z3) / 2
            print(f"{z1:+3d} {z2:+3d} {z3:+3d}   {cut:.0f}")

Two of the eight rows have a cut of 0: everyone on one side. The other six rows have one node split off from the other two, and a cut of 2. No row reaches 3.

### A graph becomes an Ising Hamiltonian

*Slide 15.*

- **Spins for nodes.** Node $i$ becomes one qubit. Its side of the cut is the value $z_i = \pm 1$ of $Z_i$.
- **Couplings for edges.** Each edge becomes one term $\frac{1 - Z_i Z_j}{2}$: 1 when the two spins differ, 0 when they match.
- **Energy is the score.** $H_C = \sum_{(i,j) \in E} \frac{1 - Z_i Z_j}{2}$ gives every bitstring an energy equal to its cut.

A sum of $Z_i Z_j$ terms is an Ising model. The best cut is the lowest-energy state of that sum.

To put the problem on a quantum computer, give every node one qubit. A measured bitstring then says where every node is: a qubit that reads 0 is on one side, and a qubit that reads 1 is on the other.

The link to the numbers $z_i$ is the Z gate, used here not as a gate but as a quantity that can be measured. $Z_i$ means "Z on qubit $i$". Its value is $+1$ when the qubit reads 0 and $-1$ when it reads 1, because Z leaves $\lvert 0 \rangle$ alone and multiplies $\lvert 1 \rangle$ by $-1$. Replacing each $z_i$ in the cost function by $Z_i$ turns the cost function into an operator: a rule that acts on states, which can be written as a table of numbers, a matrix. This operator is simple. It assigns a number to every bitstring.

Now the vocabulary of the slide. A Hamiltonian is the operator whose values are the energies of a system. Physicists call a quantity with the two values $\pm 1$ a spin, a term $Z_i Z_j$ that ties two spins together a coupling, and a set of spins with such couplings an Ising model. Nothing in this problem is a physical energy. The words are borrowed because the mathematics is the same, and "energy" here means "the score of a bitstring".

The slide speaks of two operators: $H_C$ on the third card and the Ising sum in the line under the cards.

- $H_C$, the operator on the third card, gives every bitstring an energy equal to its cut. The best cut is the bitstring with the **highest** energy of $H_C$.
- The line under the cards is about the plain sum of the $Z_i Z_j$ terms, the Ising sum $\sum_{(i,j) \in E} Z_i Z_j$. The best cut is the bitstring with the **lowest** energy of the Ising sum.

To see that the two agree, take the constant out of $H_C$. Here $\lvert E \rvert$ is the number of edges.

$$H_C = \sum_{(i,j) \in E} \frac{1 - Z_i Z_j}{2} = \frac{\lvert E \rvert}{2} - \frac{1}{2} \sum_{(i,j) \in E} Z_i Z_j$$

The same holds for the numbers: the cut value is $C = \frac{\lvert E \rvert}{2} - \frac{1}{2} \sum_{(i,j) \in E} z_i z_j$. A large cut means a small Ising sum. On the triangle, everyone on one side gives the Ising sum $1 + 1 + 1 = 3$ and the cut $\frac{3}{2} - \frac{3}{2} = 0$. One node split off gives the Ising sum $1 - 1 - 1 = -1$ and the cut $\frac{3}{2} + \frac{1}{2} = 2$.

Tonight's lab follows Qiskit's habit of looking for low energies. Its cost operator is the Ising sum, named `cost` in the code, and QAOA tries to make its value low. Whenever you read "energy" or "cost" from here on, check which operator is meant: for $H_C$ high is good, and for the Ising sum low is good.

The cell below builds $H_C$ for the triangle in Qiskit and prints the energy it gives to each of the eight bitstrings. `SparsePauliOp` is Qiskit's class for operators that are sums of products of I, X, Y and Z. These four are called the Pauli operators, and I is the identity, the operator that changes nothing. Each term is written as a label with one letter per qubit, and a number in front of it. As in a bitstring, the right-most letter belongs to qubit 0.

In [ ]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

# H_C for the triangle: 3/2, minus one half of Z Z for each of the three edges.
# "IZZ" is Z on qubits 0 and 1, "ZZI" is Z on qubits 1 and 2, "ZIZ" is Z on qubits 0 and 2.
triangle_cut = SparsePauliOp.from_list([("III", 1.5), ("IZZ", -0.5), ("ZZI", -0.5), ("ZIZ", -0.5)])
matrix = triangle_cut.to_matrix()                   # the operator as a table of 8 by 8 numbers
print("everything off the diagonal is zero:", np.allclose(matrix, np.diag(np.diag(matrix))))
print("bitstring  energy")
for k in range(8):
    print("   " + format(k, "03b") + "       " + str(int(round(matrix[k, k].real))))

The matrix has numbers only on its diagonal, one for each bitstring. Such an operator is called diagonal. It does not turn one bitstring into another. It only gives each one a score. The eight energies are the eight cut values of the triangle check: 0 for `000` and `111`, and 2 for the other six. In this list a bit 0 stands for $z = +1$ and a bit 1 for $z = -1$, and the nodes that the slide numbers 1, 2, 3 are the qubits 0, 1, 2.

### QAOA turns the cost function into a circuit

*Slide 16.*

$$\lvert \gamma , \beta \rangle = e^{- i \beta H_M} e^{- i \gamma H_C} \lvert + \rangle^{\otimes n}$$

The cost term is the cost function with each $z_i$ replaced by a Z gate on qubit $i$. The mixer term is the sum of X on every qubit.
Repeat the pair $p$ times, with new angles each time.

QAOA, the quantum approximate optimization algorithm, is the three steps of slide 5 with two adjustable angles, $\gamma$ and $\beta$. Read the formula from right to left, because the part next to the ket acts first.

- $\lvert + \rangle^{\otimes n}$ is the start. Every one of the $n$ qubits is in the state $\lvert + \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}$, which is what an H gate on every qubit makes. The symbol $\otimes n$ means "the same on each of the $n$ qubits". All $2^n$ bitstrings have the same amplitude. This is the "spread" step.
- $e^{-i\gamma H_C}$ is the cost layer. $H_C$ only gives every bitstring a score, so this layer multiplies the amplitude of each bitstring by the phase $e^{-i\gamma C}$, where $C$ is the score of that bitstring. Good cuts get a different phase from bad ones, and no probability changes. This is the "mark with a phase" step. The angle $\gamma$ sets how strongly the scores are turned into phases.
- $e^{-i\beta H_M}$ is the mixer layer. $H_M$ is the sum of X on every qubit, and the layer rotates every qubit by an angle that $\beta$ sets. This rotation moves amplitude between bitstrings that differ in one bit, so the phases from the cost layer meet and interfere. This is the "interfere" step.

A number raised to an operator, as in $e^{-i\gamma H_C}$, is how physics writes "let a system with the energy $H_C$ evolve for the time $\gamma$". For tonight it is enough to know what each of the two layers does to the amplitudes.

The pair of layers can be repeated $p$ times, each time with its own two angles, which gives $2p$ angles in all. More layers can give better answers, at the price of a longer circuit. Tonight uses $p = 1$: one cost layer, one mixer layer, two angles.

QAOA differs from Grover's search in what comes out. Grover's oracle marks one answer, and after the right number of steps that answer is nearly certain. QAOA's cost layer gives every answer a phase according to its score, and the result is a state in which good cuts are more likely than bad ones. You then measure many times and keep the best bitstring you saw. Finding good values for the angles is a job for a classical computer, as in VQE below. In tonight's lab the angles are supplied.

A word on signs. Papers and software differ in which operator they call the cost and in the signs of the angles. The slide writes the formula with $H_C$. The lab's code hands Qiskit the Ising sum from the slide before. The derivation below shows that the two give the same gates, with the angle $\gamma$ relabeled. Where they differ, follow the notebook: the cost operator is the Ising sum, and low is good.

#### Derivation: From cost function to gates

*Slide 17.*

Cost Hamiltonian: each variable becomes a Z operator.

$$H_C = \sum_{(i,j) \in E} \frac{1 - Z_i Z_j}{2}$$

Mixer Hamiltonian. $X_i$ means X on qubit $i$.

$$H_M = \sum_i X_i$$

Cost layer, as gates. All the terms of $H_C$ are diagonal, so the layer for the whole sum is the product of one small layer per term: one gate for every edge. That gate is a standard two-qubit gate called a ZZ rotation, $R_{ZZ}(\theta) = e^{-i\frac{\theta}{2} Z_i Z_j}$. It multiplies the amplitude by $e^{-i\theta/2}$ when the two qubits are equal and by $e^{+i\theta/2}$ when they differ.

$$e^{-i\gamma H_C}: \quad R_{ZZ} \text{ on every edge}$$

Which angle goes into $R_{ZZ}$ depends on the operator. With the slide's $H_C$, split off the constant as in the section before:

$$e^{-i\gamma H_C} = e^{-i\gamma \lvert E \rvert / 2} \prod_{(i,j) \in E} e^{+i\frac{\gamma}{2} Z_i Z_j}$$

The symbol $\prod$ means a product, here one factor for every edge. The factor in front is a number of size 1 that multiplies the whole state. It is a global phase and can be dropped. Each remaining factor is $R_{ZZ}(-\gamma)$.

The lab hands Qiskit the Ising sum as the cost operator. Then every factor is $e^{-i\gamma Z_i Z_j}$, which is $R_{ZZ}(2\gamma)$:

$$e^{-i\gamma \sum_{(i,j) \in E} Z_i Z_j} = \prod_{(i,j) \in E} e^{-i\gamma Z_i Z_j} = \prod_{(i,j) \in E} R_{ZZ}(2\gamma)$$

So the lab's circuit has $R_{ZZ}(2\gamma)$ on every edge. The two versions are the same gates with the angle renamed: the lab's $\gamma$ is the slide's $\gamma$ times $-\frac{1}{2}$.

Mixer layer, as gates. The X operators on different qubits do not affect each other, so this layer is one gate on every qubit. The gate $R_X(\theta) = e^{-i\frac{\theta}{2} X}$ is called an X rotation. With $\theta = \pi$ it is the X gate, up to a global phase, and with a smaller angle it is a partial flip that leaves the qubit in a superposition. In this notation $e^{-i\beta X}$ is $R_X(2\beta)$.

$$e^{-i\beta H_M}: \quad R_X(2\beta) \text{ on every qubit}$$

Count the gates: after the H gates at the start, a circuit with one layer has one two-qubit gate for every edge and one single-qubit rotation for every node. For a graph with $\lvert E \rvert$ edges that is $\lvert E \rvert$ two-qubit gates before transpiling.

The cell below builds this circuit for the triangle with `qaoa_ansatz`, the Qiskit function that Lab 2 uses. An ansatz is a circuit with adjustable angles: a template for a whole family of states. The cost operator is the Ising sum of the triangle.

In [ ]:
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp

triangle_zz = SparsePauliOp.from_list([("IZZ", 1.0), ("ZZI", 1.0), ("ZIZ", 1.0)])   # the Ising sum of the triangle
triangle_qaoa = qaoa_ansatz(triangle_zz, reps=1)                                      # reps=1 means one layer, p = 1
print("gates:", dict(triangle_qaoa.count_ops()))
print("parameters, in order:", [parameter.name for parameter in triangle_qaoa.parameters])
triangle_qaoa.draw("mpl")

The circuit has three H gates, three `rzz` gates, one for each edge, and three `rx` gates, one for each node. Its two adjustable angles are called parameters, and Qiskit lists them in the order `β[0]`, `γ[0]`: beta first, then gamma. Remember this order for Lab 3. In the drawing the ZZ rotations carry the angle `2*γ[0]` and the X rotations carry `2*β[0]`, as in the derivation.

The next cell follows the three steps of slide 5 through this circuit. It builds the circuit gate by gate with $\beta = 1.20$ and $\gamma = 0.32$, the two values that Lab 3 uses, and prints the phase and the probability of every bitstring after the cost layer, and the probability after the mixer layer. The phase is printed in units of $\gamma$.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

beta, gamma = 1.20, 0.32
by_hand = QuantumCircuit(3)
by_hand.h([0, 1, 2])                             # spread: the same amplitude on all eight bitstrings
for i, j in [(0, 1), (1, 2), (0, 2)]:
    by_hand.rzz(2 * gamma, i, j)                 # cost layer: one ZZ rotation for each edge
after_cost = Statevector(by_hand)
by_hand.rx(2 * beta, [0, 1, 2])                  # mixer layer: one X rotation on each qubit
after_mixer = Statevector(by_hand)

ising_sums = np.diag(triangle_zz.to_matrix()).real              # the value of the Ising sum for each bitstring
print("bitstring  Ising sum  cut   after the cost layer         after the mixer")
print("                            phase / gamma  probability   probability")
for k in range(8):
    cut = int(round((3 - ising_sums[k]) / 2))
    phase = np.angle(after_cost.data[k]) / gamma                # the direction of the amplitude, in units of gamma
    print(f"   {format(k, '03b')}        {ising_sums[k]:+.0f}       {cut}       {phase:+.1f}          "
          f"{after_cost.probabilities()[k]:.3f}        {after_mixer.probabilities()[k]:.3f}")

After the cost layer every bitstring still has the probability 0.125, one eighth. Only the phases have changed: each amplitude was turned by $-\gamma$ times the Ising sum of its bitstring, which is $-3\gamma$ for `000` and `111` and $+\gamma$ for the other six. That is "mark with a phase". After the mixer layer the two worst bitstrings, `000` and `111` with a cut of 0, are down to a probability of 0.007 each, and the six best ones have 0.164 each. A measurement now returns one of the best cuts in more than 98 of 100 shots. That is "interfere".

A triangle is a very small problem, and one layer handles it well. On the 12-node graph of the lab the same two angles shift the odds toward good cuts much less sharply. Lab 3 shows by how much.

### VQE is the same loop with a different cost

*Slide 18.*

$$E(\theta) = \langle \psi(\theta) \rvert H \lvert \psi(\theta) \rangle \geq E_0$$

- Any trial state gives an energy at or above the true minimum.
- A quantum computer estimates the energy; a classical optimizer updates $\theta$.
- QAOA is this loop with a circuit shaped by the problem.

Two letters change their meaning here. From this slide on, $H$ is a Hamiltonian, the operator whose values are the energies of a system. It is no longer the Hadamard gate. And $\theta$ is no longer Grover's angle. It now stands for the whole list of adjustable angles of a circuit.

VQE, the variational quantum eigensolver, looks for the lowest energy that a system can have. That energy is called the ground-state energy and is written $E_0$. The state that has it is called the ground state. The pieces of the formula:

- $\lvert \psi(\theta) \rangle$ is the trial state: the state that a circuit with the adjustable angles $\theta$ prepares.
- $\langle \psi(\theta) \rvert H \lvert \psi(\theta) \rangle$ is the expectation value of $H$ in that state: the average energy you would find if you prepared the state and measured its energy many times. The symbol $\langle \psi \rvert$ is called a bra. It is the same state as the ket $\lvert \psi \rangle$, written as a row of numbers where the ket is a column. A row times a table times a column is a single number, so the whole expression is one number, $E(\theta)$, which depends on the angles.
- $\geq E_0$ is the key fact: no trial state has an average energy below the lowest energy. The derivation below proves it.

That fact turns the search for $E_0$ into a loop. The picture on the slide shows the loop: the quantum computer estimates the energy, and the classical optimizer picks new parameters.

1. The quantum computer prepares the trial state for the current angles and estimates its energy.
2. A classical optimizer, an ordinary program that searches for the lowest value of a function, looks at that energy and proposes new angles.
3. The two take turns until the energy stops going down.

Because the energy can never go below $E_0$, a lower energy is always a better one, and the lowest value the loop finds is an upper bound on $E_0$: the true value is at or below it.

Algorithms of this kind are called variational, or hybrid, because a quantum computer and a classical computer share the work. They suit today's noisy processors, the ones whose gates and measurements go wrong now and then: each circuit is short, and the classical computer does the rest. QAOA is the same loop. Its trial circuit is shaped by the problem, its angles are $\gamma$ and $\beta$, and its energy is the expectation value of the cost operator. In tonight's lab that loop has already been run for you, which is why the two angles are supplied.

#### Derivation: Why any trial state bounds the energy

*Slide 19.*

Expand in the eigenstates of $H$. Every Hamiltonian has special states with a definite energy: measure the energy of such a state and you get the same number every time. These states are called eigenstates, and their energies are called eigenvalues. Write $\lvert E_k \rangle$ for the eigenstate with the energy $E_k$, and number them so that $E_0$ is the lowest. Any state can be written as a superposition of the eigenstates with amplitudes $c_k$. As always, the squared sizes of the amplitudes add up to one.

$$\lvert \psi \rangle = \sum_k c_k \lvert E_k \rangle, \quad \sum_k \lvert c_k \rvert^2 = 1$$

The energy is a weighted average. Measure the energy of $\lvert \psi \rangle$ and you get $E_k$ with the probability $\lvert c_k \rvert^2$. The average over many measurements is every energy times its probability, added up.

$$E = \langle \psi \rvert H \lvert \psi \rangle = \sum_k \lvert c_k \rvert^2 E_k$$

Every term is at least the minimum. Each $E_k$ is at least $E_0$, and the weights $\lvert c_k \rvert^2$ are never negative. Replacing every $E_k$ by $E_0$ can therefore only make the sum smaller, or leave it the same. $E_0$ then moves in front of the sum, and the weights add up to 1.

$$\sum_k \lvert c_k \rvert^2 E_k \geq E_0 \sum_k \lvert c_k \rvert^2 = E_0$$

This is the variational principle. In plain words: an average can never be below the smallest of the values that go into it. And an average close to the smallest value needs most of its weight on that value, so a trial state with an energy close to $E_0$ is close to the ground state, as long as the next energy $E_1$ lies well above $E_0$. That is why lowering $E$ is a way to find the ground state.

The cell below tries the principle out with numbers. It borrows the Hamiltonian of the next slides, a hydrogen molecule on two qubits. Its energies are in hartree, the unit of energy used in chemistry calculations. For now, treat it as a table of 4 by 4 numbers. `np.linalg.eigvalsh` computes the eigenvalues of such a table exactly, lowest first. The cell then computes the energy of 300 random two-qubit states. `expectation_value` computes $\langle \psi \rvert H \lvert \psi \rangle$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit.quantum_info import SparsePauliOp, random_statevector

# The Hamiltonian of the hydrogen molecule from the next slides: five terms on two qubits.
hydrogen = SparsePauliOp.from_list([("II", -1.052373), ("IZ", 0.397937), ("ZI", -0.397937),
                                    ("ZZ", -0.011280), ("XX", 0.180931)])
levels = np.linalg.eigvalsh(hydrogen.to_matrix())          # the four energies E_k, lowest first
lowest = float(levels[0])

trial_energies = []
for seed in range(300):                                    # 300 random two-qubit states
    state = random_statevector(4, seed=seed)               # the seed fixes the random numbers
    trial_energies.append(float(state.expectation_value(hydrogen).real))

print("lowest energy E_0:                    ", round(lowest, 4))
print("lowest of the 300 trial energies:     ", round(min(trial_energies), 4))
print("highest of the 300 trial energies:    ", round(max(trial_energies), 4))
print("every trial energy is at or above E_0:", all(energy_k >= lowest for energy_k in trial_energies))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(trial_energies, bins=30, color="tab:blue", edgecolor="white")
ax.axvline(lowest, color="tab:red", ls="--", label="lowest energy E_0")
ax.set_xlabel("energy of the trial state (hartree)")
ax.set_ylabel("number of states")
ax.set_title("Energies of 300 random trial states")
ax.legend()
plt.show()

The lowest of the 300 trial energies is $-1.7768$ and the highest is $-0.4008$. None is below $E_0 = -1.8573$: the whole histogram, the bar chart of how many trial states fall in each range of energy, stays to the right of the dashed line. Random guessing also never comes close to the line. Even the best of the 300 guesses is 0.08 above it. Finding the minimum takes a search, and that is the optimizer's job.

### VQE finds the ground-state energy of a hydrogen molecule

*Slide 20.*

$$E(\theta) = \sum_k h_k \langle \psi(\theta) \rvert P_k \lvert \psi(\theta) \rangle$$

- At its normal bond length, $\text{H}_2$ maps to two qubits and five Pauli terms.
- The circuit prepares a trial state. The processor measures each term's average.
- The optimizer lowers the energy to $-1.857$ hartree, the exact lowest energy of this two-qubit model (electrons only).

A note on the symbols first. The slide writes the coefficients in the formula as $c_k$. This notebook writes $h_k$, because $c_k$ already stands for the amplitudes in the derivation above. And $\text{H}_2$ is the hydrogen molecule, two hydrogen atoms bound together. It is not the Hamiltonian $H$.

A hydrogen molecule has two nuclei and two electrons. Its Hamiltonian describes the energy of the two electrons while the nuclei sit at a fixed distance from each other. Chemistry has standard recipes for rewriting such a Hamiltonian as an operator on qubits. For hydrogen at its normal bond length, the usual distance between the two nuclei, the result needs only two qubits and is a sum of five terms. Each term is a number $h_k$ times a Pauli term $P_k$, a product of the operators I, X and Z with one of them for each qubit. The next section lists the five terms. The bond length used is 0.735 angstrom, and one angstrom is $10^{-10}$ meters. The recipe has three standard names, in case you want to look it up later: the STO-3G basis, a small set of functions that describes where each electron can be; the parity mapping, a rule that turns the description of the electrons into an operator on qubits; and the two-qubit reduction, which removes two of the four qubits that the mapping gives, because the number of electrons fixes their values. Tonight you only need the five terms.

The formula says how the energy is measured. A quantum computer cannot measure $H$ in one go, but it can measure the average of each Pauli term in the trial state. Multiply each average by its coefficient $h_k$ and add up: that is the energy $E(\theta)$. In Qiskit this job is done by the Estimator. The Estimator is one of Qiskit's two primitives, the standard interfaces for running circuits: you hand it a circuit, an operator and values for the angles, and it returns the expectation value. The other primitive, the Sampler, returns measured bitstrings. Lab 1 uses it.

Energies of molecules are given in hartree, the unit of energy used in chemistry calculations. Its symbol is Ha.

In the third bullet, $-1.857$ hartree is the exact lowest energy of this two-qubit model of the molecule: the lowest eigenvalue of the five-term operator. A model this small is a simplified description of the molecule, so the number is exact for the model, not for a hydrogen molecule measured in a laboratory. It also counts the energy of the electrons only. The next section adds the energy of the two nuclei.

The picture on the slide shows the energy against the optimizer step for a VQE run on a simulator, falling to the exact value. The second cell of the next section draws that plot.

#### Derivation: The hydrogen Hamiltonian, term by term

*Slide 21.*

Two qubits, five terms. Each term is written with two letters, one for each qubit. As in Qiskit's labels, the right letter acts on qubit 0 and the left letter on qubit 1. $I$ is the identity, so $IZ$ is Z on qubit 0 alone, $ZI$ is Z on qubit 1 alone, $ZZ$ is the product of the two, and $XX$ is X on both qubits. $II$ changes nothing: its value is 1 for every state, so its coefficient is a constant that is added to every energy.

$$H = -1.0524\,II + 0.3979\,IZ - 0.3979\,ZI - 0.0113\,ZZ + 0.1809\,XX$$

Energy of a trial state. The five numbers in this line are the coefficients $h_k$, and the five letter pairs are the Pauli terms $P_k$.

$$E(\theta) = \sum_k h_k \langle \psi(\theta) \rvert P_k \lvert \psi(\theta) \rangle$$

On a processor every Pauli term is measured separately. The four terms made of I and Z come from one set of shots: measure both qubits, and for every shot work out the value of the term from the two bits, with $+1$ for a 0 and $-1$ for a 1. The $XX$ term needs a second set of shots, taken after an H gate on each qubit, because H turns a measurement of Z into a measurement of X. The energy is the weighted sum of the five averages.

Exact ground state of the model, electrons only. The operator is a table of 4 by 4 numbers, and an ordinary computer finds its lowest eigenvalue directly.

$$E_0 = -1.8573 \text{ Ha}$$

Add the repulsion between the nuclei. The two nuclei both carry a positive charge and repel each other. At a fixed distance this adds a constant to the energy, one that does not depend on what the electrons do. In the units of these calculations the constant is 1 divided by the distance, with the distance measured in units of 0.5292 angstrom: $\frac{0.5292}{0.735} = 0.7200$ hartree. The sum is the total energy of the molecule in this model.

$$-1.8573 + 0.7200 = -1.1373 \text{ Ha}$$

The cell below looks at the Hamiltonian with numbers. It reuses `hydrogen` from the cell above. First it prints the energy of each of the four bitstrings, then the exact energies. `np.linalg.eigh` returns the eigenvalues together with the eigenstates.

In [ ]:
import numpy as np
from qiskit.quantum_info import Statevector

print("energy of each single bitstring, with qubit 0 as the right character:")
for label in ["00", "01", "10", "11"]:
    print("  ", label, round(float(Statevector.from_label(label).expectation_value(hydrogen).real), 4))

levels, states = np.linalg.eigh(hydrogen.to_matrix())      # exact energies, and the states that have them
ground = np.abs(states[:, 0]) ** 2                         # probabilities of 00, 01, 10, 11 in the ground state
print("the four exact energies:", np.round(levels, 4))
print("lowest energy E_0:", round(float(levels[0]), 4))
print("probabilities of 00, 01, 10, 11 in the ground state:", np.round(ground, 3) + 0.0)
print("with the repulsion between the nuclei:", round(float(levels[0]) + 0.7200, 4))

No single bitstring reaches the minimum. The best one is `01`, with the energy $-1.837$. You can check that number by hand. In `01` qubit 0 reads 1 and qubit 1 reads 0, so $IZ$ has the value $-1$, $ZI$ has $+1$, $ZZ$ has $-1$, and $XX$ averages to zero. With the coefficients from the code, $-1.052373 - 0.397937 - 0.397937 + 0.011280 = -1.836967$.

The exact lowest energy is $-1.8573$, lower by 0.02. The ground state is a superposition: mostly `01`, with the probability 0.988, and a little `10`, with the probability 0.012. The $XX$ term flips both qubits, so it connects `01` with `10`, and it is this term that rewards the superposition. No single bitstring can use it. A trial circuit for this problem therefore has to be able to make superpositions.

The last line adds the repulsion between the nuclei and gives $-1.1373$, as in the derivation.

The next cell runs the whole VQE loop and records every energy that the optimizer asks for. It is the code of the "Try it" in Part 2, with other names and one addition, the list `history`. Part 2 explains the code line by line. Here, look at the plot.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorEstimator

trial_circuit = efficient_su2(2, reps=1)        # a ready-made trial circuit with 8 adjustable angles
exact_estimator = StatevectorEstimator()        # computes the energy of a circuit exactly, without a quantum computer
history = []                                    # every energy the optimizer asks for, in order


def logged_energy(values):
    """Return the energy of the trial state for one choice of the 8 angles, and keep a record of it."""
    job = exact_estimator.run([(trial_circuit, hydrogen, values)])
    history.append(float(job.result()[0].data.evs))
    return history[-1]


outcome = minimize(logged_energy, np.full(trial_circuit.num_parameters, 0.1), method="COBYLA")
print("energy evaluations:", len(history))
print("first energy:", round(history[0], 4), "| lowest energy found:", round(float(outcome.fun), 4),
      "| exact:", round(lowest, 4))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(history, color="tab:blue", label="energy of the trial state")
ax.axhline(lowest, color="tab:red", ls="--", label="exact lowest energy")
ax.set_xlabel("energy evaluation")
ax.set_ylabel("energy (hartree)")
ax.set_title("VQE for hydrogen on two qubits")
ax.legend()
plt.show()

The first energy, for the starting angles, is $-1.0373$. During the first few dozen evaluations the energy falls steeply, with some jumps upward while the optimizer tries out directions. After about 200 evaluations the curve is within a thousandth of a hartree of the dashed line. The optimizer stops by itself after about 600 evaluations, at $-1.8573$. The exact count depends on the installed version of SciPy, the library that provides the optimizer, so the cell prints it. The curve never dips below the dashed line, which is the variational principle at work.

On a simulator the whole run takes about a second. On a real processor every evaluation is a job that waits in a queue, the line of jobs that arrived before it, so a run of several hundred evaluations belongs on a simulator. This is also why tonight's hardware lab comes with its angles already chosen.

Slide 22 is a cell to try. You find it at the start of Part 2, under "Try it: VQE for hydrogen on two qubits".

## Part 2: Hands-on

The code in the "Try it" and "Lab" cells is the code on the slides. Run each cell, read what it prints, then change something and run it again. After each lab comes a "Your turn" exercise with a check cell.

The cells of Part 2 need the Setup section and each other, in order. They do not use anything that the code cells of Part 1 define, so the lab also works if you have not run those.

### Try it: VQE for hydrogen on two qubits

*Slide 22.*

File on the slide: `vqe_h2.py`

1. Write the Hamiltonian as five Pauli terms
2. Pick a one-layer trial circuit
3. Let the optimizer lower the energy
4. Compare with $-1.857$

This cell is the whole VQE loop from Part 1 in twenty lines. Line by line:

- `SparsePauliOp.from_list([...])` builds the Hamiltonian from its five terms. Each term is a label, with one letter for each qubit, and its coefficient. The code names the Hamiltonian `H`.
- `efficient_su2(2, reps=1)` is a ready-made trial circuit for two qubits from Qiskit's library. A circuit with adjustable angles is called an ansatz, which explains the name of the variable. This one has rotations on each qubit, one CX gate between the two qubits, and rotations again: 8 adjustable angles in all. CX flips one qubit when the other one is 1. It is what lets the circuit make states such as the superposition of `01` and `10` that the ground state needs.
- `StatevectorEstimator()` is an Estimator that needs no quantum computer: it runs on the computer that runs this notebook. An Estimator takes a circuit, an operator and values for the angles, and returns the expectation value, the average of the operator in the state the circuit prepares. This one computes it exactly, with no shots and no noise.
- `energy(theta)` is the function that the optimizer will make as low as possible. `est.run([(ansatz, H, theta)])` starts a job with one item: the circuit, the operator and the angles. A job is one request to run circuits. `job.result()[0].data.evs` is the expectation value, which is the energy $E(\theta)$.
- `x0` is the starting point: all 8 angles equal to 0.1.
- `minimize(energy, x0, method="COBYLA")` is the classical optimizer, from the SciPy library. COBYLA is a search method that needs only the values of the function. It calls `energy` again and again with new angles.
- `res.fun` is the lowest energy the optimizer found.

Before you run the cell: which number should it print?

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

H = SparsePauliOp.from_list([
    ("II", -1.052373), ("IZ", 0.397937),
    ("ZI", -0.397937), ("ZZ", -0.011280),
    ("XX", 0.180931)])
ansatz = efficient_su2(2, reps=1)
est = StatevectorEstimator()

def energy(theta):
    job = est.run([(ansatz, H, theta)])
    return float(job.result()[0].data.evs)

x0 = np.full(ansatz.num_parameters, 0.1)
res = minimize(energy, x0, method="COBYLA")
print(round(res.fun, 4))   # about -1.8573

The cell prints `-1.8573`: the exact lowest energy of the model, to four decimals. That is step 4 of the slide. The run takes about a second and uses about 600 energy evaluations. The comment in the last line says "about" because an optimizer stops when it finds no further improvement, a tiny distance above the true minimum.

Things to try:

- Add `print(res.nfev)` at the end. It prints the number of energy evaluations.
- Add `print(round(res.fun + 0.7200, 4))`. It adds the repulsion between the nuclei and prints the total energy of the molecule in this model.
- Change the 0.1 in `x0` to another number. The optimizer then takes a different path and a different number of evaluations. From some starting points it stops at its limit of 1,000 evaluations, a little above the minimum. It never ends below $-1.8573$, whatever the starting point: that is the variational principle.

### Lab 1: Grover on two qubits

*Slide 24.*

File on the slide: `grover.py`

1. Build the oracle: one CZ marks 11
2. Build the diffuser
3. Run on the simulator
4. Confirm one outcome dominates

This is Grover's search for one marked result among the four results of two qubits. The marked result is 11, and one step is enough. Line by line:

- `QuantumCircuit(2)` makes a circuit with two qubits, both starting in 0.
- `qc.h([0, 1])` applies an H gate to qubit 0 and to qubit 1. All four results now have the amplitude $\frac{1}{2}$. The comment calls this "every answer at once". Read it as "an equal superposition of every answer". The speedup does not come from trying every answer at once. It comes from the interference in the diffuser.
- `qc.cz(0, 1)` is the oracle. It puts a minus sign on the amplitude of 11.
- The next five lines are the diffuser. It turns the minus sign into a difference in size: the amplitude of 11 grows to size 1 and the other three drop to 0.
- `qc.measure_all()` adds a measurement of every qubit. The measured bits are stored under the name `meas`, which is why the result is read from `result.data.meas`.
- `StatevectorSampler()` is a Sampler that needs no quantum computer: it runs on the computer that runs this notebook. A Sampler runs a circuit for a number of shots and returns the measured bitstrings. A shot is one run of the circuit. This one computes the exact state and draws the shots from it, so it behaves like an ideal quantum computer, one without noise.
- `sampler.run([qc], shots=1024)` starts a job with one circuit and 1,024 shots. `job.result()` waits for the job and returns one result for each circuit, so `[0]` picks the result of ours.
- `get_counts()` turns the measured bits into a dictionary that maps each bitstring to its number of shots.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

qc = QuantumCircuit(2)
qc.h([0, 1])          # every answer at once
qc.cz(0, 1)           # oracle: mark 11
qc.h([0, 1])          # diffuser starts
qc.x([0, 1])
qc.cz(0, 1)
qc.x([0, 1])
qc.h([0, 1])          # diffuser ends
qc.measure_all()

sampler = StatevectorSampler()
job = sampler.run([qc], shots=1024)
result = job.result()[0]
print(result.data.meas.get_counts())
# {'11': 1024}

All 1,024 shots give `11`, as the last comment in the cell says. The cell prints the same line every time you run it, because the probability of `11` is exactly 1. One use of the oracle was enough to find the marked result among four.

This certainty belongs to the ideal simulator. On a real processor a few shots would land on the other three results.

#### Your turn 1

Change the oracle so that Grover's search finds `01`. In `01`, qubit 0 reads 1 and qubit 1 reads 0, because Qiskit prints qubit 0 on the right. The cell below already has the H gates at the start and the diffuser at the end. Write your oracle in place of the line `# YOUR CODE HERE`. Gates from tonight are enough. You do not need to add measurements, because the check cell computes the probabilities from the circuit.

In [ ]:
from qiskit import QuantumCircuit

grover01 = QuantumCircuit(2)
grover01.h([0, 1])          # equal amplitudes on all four results
# YOUR CODE HERE
grover01.h([0, 1])          # the diffuser, as in Lab 1
grover01.x([0, 1])
grover01.cz(0, 1)
grover01.x([0, 1])
grover01.h([0, 1])
grover01.draw("mpl")

In [ ]:
from qiskit.quantum_info import Statevector

assert grover01.size() > 11, "The oracle is still missing. Replace the line YOUR CODE HERE with gates that mark 01."
body = grover01.remove_final_measurements(inplace=False)
assert "measure" not in body.count_ops(), "Take the measurement out of your oracle. The check computes the probabilities from the circuit, so the circuit needs no measurements."
p = Statevector(body).probabilities_dict()
assert p.get("10", 0) < 0.99, "You marked 10. Qiskit prints qubit 0 on the right, so in 01 it is qubit 0 that reads 1."
assert p.get("11", 0) < 0.99, "This still marks 11, as in the lab. A CZ marks 11, with or without an X gate in front of it, because that X gate meets four equal amplitudes and changes nothing. Make the result you want look like 11 to the CZ, then undo that change."
assert p.get("01", 0) > 0.99, "The CZ puts its minus sign only where both qubits are 1. Use X gates so that the result you want looks like 11 to it."
print("Check passed: the search ends in 01 with probability", round(float(p["01"]), 3))

### Lab 2: from the 12-node graph to a circuit

*Slide 25.*

File on the slide: `maxcut_qaoa.py`

1. Load the graph's edge list
2. Write the cost operator
3. Build the $p = 1$ QAOA circuit
4. Count its two-qubit gates before and after transpiling

Labs 2 and 3 are one experiment: Max-Cut on a graph with 12 nodes and 16 edges, solved with the QAOA circuit from Part 1. Lab 2 builds the circuit and Lab 3 runs it.

The code on the slide uses two names that it does not define: the edge list `edges` and the pass manager `pm`. The next cell provides them. It is not on the slide. What it does:

- `edges` is the graph, as a list of 16 pairs of nodes. This is step 1 of the slide.
- `connect("ibm")` and `least_busy` pick a backend, as described in the Setup section. A backend is the machine, real or simulated, that runs a circuit. Unless you switched to real hardware, it is the simulated IBM processor.
- A processor cannot run every circuit as it is written. It has a small set of gates of its own, called its native gates, and a two-qubit gate works only between qubits that are neighbors on the chip. Rewriting a circuit so that it uses only those gates and those connections is called transpiling. In Qiskit the object that does it is called a pass manager. `generate_preset_pass_manager` builds one for a backend, and `optimization_level=1` asks it for light optimization.
- The rule book that a pass manager follows is called the target: the list of the gates a backend has, and of the qubits each gate can act on.
- The function `patch_target` and the last three lines matter only on the simulated processor. A simulation slows down with every extra qubit that a circuit touches, and the transpiler is free to spread the circuit over the whole 156-qubit chip. When it does, the job of Lab 3 can take a minute or more. So on the simulated processor the pass manager gets a smaller rule book. `patch_target` copies the target and allows the two-qubit gate only inside one ring of 12 neighboring qubits, the ones listed in `CELL`. The job then takes several seconds. The name `CELL` refers to a ring of qubits on the chip. It has nothing to do with the cells of this notebook. On a real processor none of this applies, and the transpiler uses the whole chip. You do not need to follow the code of `patch_target` to do the lab.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
from qiskit.transpiler import Target, generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

# The 12-node graph for tonight, as a list of edges (pairs of nodes).
edges = [(0, 1), (0, 4), (1, 2), (1, 5), (2, 3), (3, 6), (3, 7), (4, 5),
         (4, 8), (5, 9), (6, 7), (6, 11), (7, 11), (8, 9), (9, 10), (10, 11)]

# One 12-qubit cell of the simulated processor: a ring of 12 neighboring qubits.
CELL = [101, 102, 103, 104, 105, 116, 117, 121, 122, 123, 124, 125]


def patch_target(backend, qubits):
    """Return the backend's rule book with the two-qubit gate allowed only inside `qubits`."""
    keep = set(qubits)
    patch = Target(num_qubits=backend.num_qubits)
    for name in backend.target.operation_names:
        rows = {where: props for where, props in backend.target[name].items()
                if where is not None and (len(where) == 1 or set(where) <= keep)}
        if rows:
            patch.add_instruction(backend.target.operation_from_name(name), rows)
    return patch


# The backend and the pass manager, as in Monday's lab.
service = connect("ibm")
backend = service.least_busy(
    operational=True, simulator=False)
pm = generate_preset_pass_manager(
    optimization_level=1, backend=backend)
if isinstance(service, SimulatedService):
    pm = generate_preset_pass_manager(
        optimization_level=1, target=patch_target(backend, CELL))

The cell prints one line from `connect`, which says where the circuits of Labs 2 and 3 will run.

The next cell draws the graph. The nodes are numbered 0 to 11 and placed on a grid with three rows of four. The function `draw_graph` is used again at the end of Lab 3, to draw a cut.

In [ ]:
import matplotlib.pyplot as plt


def draw_graph(sides=None, title="Tonight's graph: 12 nodes, 16 edges"):
    """Draw the lab graph. Node k is drawn in row k // 4 and column k % 4 of a grid.

    sides is optional: a list with a 0 or a 1 for every node. With it, the two sides get two
    colors, and the edges that do not cross the cut are drawn as dotted red lines.
    """
    fig, ax = plt.subplots(figsize=(5, 3.4))
    for i, j in edges:
        crosses = sides is None or sides[i] != sides[j]
        ax.plot([i % 4, j % 4], [-(i // 4), -(j // 4)], lw=2, zorder=1,
                color="black" if crosses else "tab:red", ls="-" if crosses else ":")
    for node in range(12):
        color = "tab:blue" if sides is None else ("tab:purple" if sides[node] == 1 else "0.55")
        ax.scatter(node % 4, -(node // 4), s=420, color=color, zorder=2)
        ax.text(node % 4, -(node // 4), str(node), ha="center", va="center", color="white", zorder=3)
    ax.set_xlim(-0.4, 3.4)
    ax.set_ylim(-2.4, 0.4)
    ax.set_title(title)
    ax.axis("off")
    plt.show()


draw_graph()

Look at the right-hand side of the drawing. Nodes 3, 6 and 7 form a triangle, and so do nodes 6, 7 and 11. Part 1 showed that a triangle can never have all three of its edges cut, so no split of this graph can cut all 16 edges.

Now the code of the slide. Line by line:

- `n = 12`: one qubit for each node.
- The line that starts with `# edges =` is a comment. It is a reminder that the edge list comes from the notebook, and you have already run the cell that holds it.
- `SparsePauliOp.from_sparse_list(...)` builds the cost operator. For every edge `(i, j)` it adds the term `("ZZ", [i, j], 1.0)`: Z on qubit `i` times Z on qubit `j`, with the coefficient 1.0. This is the Ising sum from Part 1, for which a low value means a large cut. `SparsePauliOp` is Qiskit's class for operators that are sums of such terms. This is step 2 of the slide.
- `qaoa_ansatz(cost, reps=1)` builds the QAOA circuit with one layer, $p = 1$: an H gate on every qubit, a ZZ rotation for every edge, and an X rotation on every qubit. An ansatz is a circuit with adjustable angles. This one has two, $\beta$ and $\gamma$, and they stay open until Lab 3. This is step 3.
- `qc.measure_all()` adds a measurement of every qubit.
- `isa = pm.run(qc)` transpiles the circuit for the backend. The comment says that `pm` comes from Monday's lab. In this notebook it comes from the cell above. A circuit written in a backend's own gates is called an ISA circuit, for "instruction set architecture", which explains the name `isa`.
- `isa.count_ops()` counts the gates of the transpiled circuit by name. This is the second half of step 4.

In [ ]:
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp

n = 12
# edges = [(0, 1), (0, 4), ...]  from the notebook

cost = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], 1.0) for i, j in edges],
    num_qubits=n)

qc = qaoa_ansatz(cost, reps=1)
qc.measure_all()

isa = pm.run(qc)      # pm from Monday's lab
print(isa.count_ops())

The cell prints the gate counts after transpiling, as an `OrderedDict`, a dictionary that keeps its entries in a fixed order. Depending on your version of Python it is shown with curly brackets or as a list of pairs. On the simulated processor, and on the IBM processors checked for this notebook, the native gates include `rz`, `sx` and `x` on single qubits and `cz` on pairs of qubits. The entry to look at is `cz`, the two-qubit gate. On the simulated processor it is 53. The counts of `rz` and `sx` change a little from one run to the next, because the transpiler makes some of its choices at random. On a real processor all the numbers can be different, because the transpiler works with the whole chip.

Step 4 also asks for the count before transpiling. The cell of the slide cannot print it any more, because `qc` now also holds the measurements. The next cell builds the circuit once more, without measurements, and prints both counts side by side.

In [ ]:
before = qaoa_ansatz(cost, reps=1)          # the same circuit, before measuring and before transpiling
print("before transpiling:", dict(before.count_ops()))
print("after transpiling: ", dict(isa.count_ops()))
print("parameters, in order:", [parameter.name for parameter in qc.parameters])

Before transpiling the circuit has 16 `rzz` gates, one ZZ rotation for each edge of the graph, plus 12 H gates and 12 `rx` gates, one of each for every node. After transpiling it has more two-qubit gates: 53 on the simulated processor. They come from two places.

- The processor has no `rzz` gate. Each one is rebuilt from two `cz` gates and some single-qubit gates. That makes 32.
- The edges of the graph do not match the connections of the processor. In the ring of 12 qubits every qubit has two neighbors, while most nodes of the graph have three. When two qubits need a gate and are not neighbors, the transpiler first moves their states next to each other with SWAP gates, and each SWAP costs three `cz` gates. Seven SWAPs make the other 21.

Every extra gate is one more chance for an error. Friday's session is about that, and so is Friday's challenge.

The last line of the output repeats what Part 1 found: the circuit has two parameters, and their order is `β[0]`, then `γ[0]`.

#### Your turn 2

Build the cost operator and the one-layer QAOA circuit for the triangle from Part 1, the way Lab 2 built them for the 12-node graph. The triangle has three nodes, 0, 1 and 2, and its edges are already in the list `tri_edges`. Store the operator in `tri_cost` and the circuit in `tri_qc`. Use the Ising sum, as the lab does: one ZZ term for each edge, with the coefficient 1.0.

Keep to the names `tri_edges`, `tri_cost` and `tri_qc`, and do not assign `n`, `edges`, `cost`, `qc` or `isa` here. The cells after this exercise still need those five as Lab 2 left them, for the 12-node graph.

In [ ]:
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp

tri_edges = [(0, 1), (1, 2), (0, 2)]      # the triangle: three nodes, three edges
tri_cost = None
tri_qc = None
# YOUR CODE HERE
print("tri_cost:", tri_cost)
print("gates in tri_qc:", dict(tri_qc.count_ops()) if hasattr(tri_qc, "count_ops") else tri_qc)

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

assert tri_cost is not None and tri_qc is not None, "Build tri_cost and tri_qc in the cell above, in place of the line YOUR CODE HERE."
lab_names_changed = []                   # the later cells still need the names of Lab 2 for the 12-node graph
if globals().get("n", 12) != 12:
    lab_names_changed.append("n")
if len(globals().get("edges", [None] * 16)) != 16:
    lab_names_changed.append("edges")
for name in ("cost", "qc"):
    if getattr(globals().get(name), "num_qubits", 12) == 3:
        lab_names_changed.append(name)
if getattr(globals().get("isa"), "num_clbits", 12) != 12:      # Lab 2 measures all 12 qubits
    lab_names_changed.append("isa")
assert not lab_names_changed, ("The cells after this exercise still need " + ", ".join(lab_names_changed) + " for the 12-node graph. "
                               "Give the triangle its own names, then run the cells of Lab 2 again.")
assert isinstance(tri_cost, SparsePauliOp), "tri_cost should be a SparsePauliOp. Look at how Lab 2 builds cost."
assert isinstance(tri_qc, QuantumCircuit), "tri_qc should be a circuit. Look at how Lab 2 builds qc."
assert tri_cost.num_qubits == 3 and tri_qc.num_qubits == 3, "The triangle has three nodes, so the operator and the circuit need three qubits."


def ising_value(bits):
    """The value that tri_cost gives to one bitstring."""
    return float(Statevector.from_label(bits).expectation_value(tri_cost).real)


assert abs(ising_value("000") - 3) < 1e-9, "With all three nodes on one side, every edge should add +1 to the cost."
for bits in ["001", "010", "011", "100", "101", "110"]:
    assert abs(ising_value(bits) + 1) < 1e-9, "Use one ZZ term for each of the three edges, each with the coefficient 1.0."
body = tri_qc.remove_final_measurements(inplace=False)
assert body.num_parameters == 2, "tri_qc should have exactly two parameters, one beta and one gamma, that still have no value: the check sets them itself. Use reps=1, and do not assign the angles in this cell."
flat = Statevector(body.assign_parameters([0, 0])).probabilities()
assert np.allclose(flat, 1 / 8), "With both angles at zero every bitstring should be equally likely. Build the circuit with qaoa_ansatz."
at_lab_angles = float(Statevector(body.assign_parameters([1.20, 0.32])).expectation_value(tri_cost).real)
assert abs(at_lab_angles + 0.9434) < 1e-3, "Build tri_qc with qaoa_ansatz from tri_cost, the operator with all three edges."
lowest_cost, best_angles = 9.0, None
for beta_try in np.arange(0.2, 3.01, 0.2):
    for gamma_try in np.arange(0.2, 3.01, 0.2):
        state = Statevector(body.assign_parameters([beta_try, gamma_try]))
        value = float(state.expectation_value(tri_cost).real)
        if value < lowest_cost:
            lowest_cost, best_angles = value, (round(float(beta_try), 1), round(float(gamma_try), 1))
assert lowest_cost < -0.5, "Changing the angles should lower the average cost. Build the circuit from tri_cost."
print("Check passed.")
print("lowest average cost on a grid of angles:", round(lowest_cost, 2), "at [beta, gamma] =", list(best_angles))
print("average cut at those angles:", round((3 - lowest_cost) / 2, 2), "of 3 edges")

### Lab 3: sample on hardware, score every bitstring

*Slide 26.*

File on the slide: `score_cuts.py`

1. Bind the supplied angles
2. Submit to hardware
3. Compute the cut value of each sampled bitstring
4. Report the best one

The circuit from Lab 2 still has two open angles. This lab fills them in, runs the circuit 4,096 times, and looks for the best cut among the bitstrings that were measured.

The angles are supplied, so that nobody has to run an optimizer during the session. They were found beforehand by a classical search over both angles. The order matters: the circuit lists its parameters as beta first and gamma second, so `[1.20, 0.32]` means $\beta = 1.20$ and $\gamma = 0.32$. The next cell holds them. It is not on the slide.

In [ ]:
angles = [1.20, 0.32]   # [beta, gamma], the order of qc.parameters

The code of the slide, line by line:

- `cut_value(bits, edges)` scores one measured bitstring. Qiskit prints qubit 0 as the last character, so `bits[::-1]` first reverses the string. After that, `z[i]` is the bit of node `i`. The sum counts the edges whose two ends hold different bits, because Python counts `True` as 1. That count is the cut value. This is step 3 of the slide.
- `isa.assign_parameters(angles)` fills the two angles into the transpiled circuit. Giving a parameter its value is called binding it. This is step 1.
- `SamplerV2(mode=backend)` is the Sampler for IBM backends. It comes from the package qiskit-ibm-runtime, and it was imported in the cell that built the pass manager. `sampler.run([bound], shots=4096)` submits the job. This is step 2.
- `counts` maps each measured bitstring to its number of shots.
- `max(counts, key=...)` picks, among the bitstrings that were measured, the one with the largest cut value, and the last line prints it with its cut value. The part `lambda b: cut_value(b, edges)` is a short way to write a small function: it takes a bitstring `b` and returns its cut value. This is step 4.

With `USE_HARDWARE = False` everything happens on the computer that runs this notebook, and nothing is sent. The backend is the simulated processor, which imitates the errors of the IBM processor ibm_fez, and the cell takes several seconds: between 7 and 15 on the laptop used to test this notebook. On a laptop that let the simulator use only two of its cores, the parts of a computer chip that run calculations side by side, it took about half a minute, so on a small computer give it time. With `USE_HARDWARE = True` the job goes to a real IBM processor and waits in its queue, the line of jobs that arrived before yours. When this cell was run on the IBM processor ibm_fez on Nov 13, 2026, the job was done about 15 seconds after it started and used about 3 seconds of quantum time, the time a real processor spends running your job. That is one measurement. Queues vary.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
def cut_value(bits, edges):
    z = bits[::-1]    # qubit 0 is the last bit
    return sum(z[i] != z[j] for i, j in edges)

bound = isa.assign_parameters(angles)
sampler = SamplerV2(mode=backend)
job = sampler.run([bound], shots=4096)
result = job.result()[0]
counts = result.data.meas.get_counts()

best = max(counts,
           key=lambda b: cut_value(b, edges))
print(best, cut_value(best, edges))

The cell prints the best bitstring among your samples and its cut value. In every test run on the simulated processor the cut value was 15, and the bitstring was `010111100101` or `101000011010`. These two are mirror images: every 0 of one is a 1 of the other. They put the same nodes together and name the two sides the other way round, so they are the same cut.

**If your job is stuck in the queue.** Interrupt the kernel (Jupyter and qBraid Lab: Kernel menu, Interrupt Kernel. Google Colab: Runtime menu, Interrupt execution). Then change `USE_CACHED` to `True` in the next cell and run it. It loads a result that was recorded on a real IBM processor, so the rest of the notebook works. It also needs `cut_value`, which the lab cell above defines. If the kernel was restarted after you started your job, set `USE_HARDWARE = False` in the first code cell, run the three Setup cells, then run the cells of Lab 2 and Lab 3 in order, starting with the cell that defines `edges` and `pm`, down to the lab cell above. That cell then runs on the simulator, which takes several seconds as described above, and sends nothing, and after it the cached result loads. Interrupting stops the waiting, not the job: the job stays in IBM's queue. If your job finished, leave `USE_CACHED = False`.

The recorded run had 1,566 different bitstrings among its 4,096 shots. To keep the cell short, it holds the 200 most frequent ones, which cover 1,762 of the shots. If you load it, the numbers further down refer to those 1,762 shots. Because the rare bitstrings are left out, the average cut of this selection, 11.96, is higher than that of the full run, 10.49, and higher than a circuit without noise would give. That is an effect of the selection, not of the processor.

In [ ]:
# Cached result for the cell above, in case your job is still waiting in the queue.
# Recorded Nov 13, 2026 on the IBM processor ibm_fez (4,096 shots, 1,566 different bitstrings).
# The cell holds the 200 most frequent bitstrings of that run: 1,762 of the 4,096 shots.
CACHED_QAOA = {
    "010111100101": 37, "010110100101": 27, "101000011010": 27, "010110101101": 24, "101000011101": 22, "101010011010": 22,
    "010110101010": 21, "100101100101": 20, "010110101001": 19, "011011010101": 18, "100100101101": 18, "101010010110": 18,
    "010110011010": 17, "100110100101": 17, "101001011010": 17, "110100101101": 17, "110110100101": 17, "010101101101": 16,
    "011011010110": 16, "101000011110": 16, "011010011010": 15, "011001011010": 14, "011011010010": 14, "101001010101": 14,
    "010110010110": 13, "101000101001": 13, "101110100101": 13, "001001011010": 12, "001010011010": 12, "010101100101": 12,
    "010110101011": 12, "010111010110": 12, "101000001011": 12, "101001100101": 12, "101010010010": 12, "101010100101": 12,
    "110100011010": 12, "110100101001": 12, "110110101101": 12, "001011010110": 11, "001111000101": 11, "010010110100": 11,
    "010010111010": 11, "010100100101": 11, "010110100100": 11, "010111100011": 11, "011010010110": 11, "101000011100": 11,
    "101000101101": 11, "101001010110": 11, "101110101101": 11, "110100101010": 11, "000110011010": 10, "001011010010": 10,
    "001111100101": 10, "010010010110": 10, "010101001011": 10, "010111100001": 10, "011010010101": 10, "100110101010": 10,
    "100110101101": 10, "101000111000": 10, "101010010100": 10, "101010010101": 10, "101010110100": 10, "101100101101": 10,
    "110100101011": 10, "010010011010": 9, "010011100101": 9, "010011110100": 9, "010101010010": 9, "010101101001": 9,
    "010111010010": 9, "010111101101": 9, "011010011001": 9, "101000010110": 9, "101010011110": 9, "111010011010": 9,
    "001010011101": 8, "001011100101": 8, "010001011010": 8, "010101011110": 8, "010101101010": 8, "010111000110": 8,
    "010111100010": 8, "010111110100": 8, "011000011010": 8, "100100101001": 8, "100101101010": 8, "100101101101": 8,
    "101000011011": 8, "101001000101": 8, "101100001010": 8, "101100001011": 8, "101100101001": 8, "101110011010": 8,
    "110101100101": 8, "001011010101": 7, "001110101101": 7, "010010111001": 7, "010011110101": 7, "010101101011": 7,
    "010111010101": 7, "010111100111": 7, "011010010010": 7, "011010011101": 7, "100100111100": 7, "100101100100": 7,
    "100110011010": 7, "100110100010": 7, "101000101010": 7, "101010011101": 7, "101100011010": 7, "110100101100": 7,
    "001010011100": 6, "010010100101": 6, "010100101001": 6, "010110110100": 6, "010111100100": 6, "011011010100": 6,
    "011011011010": 6, "011110011010": 6, "100100101010": 6, "100101101001": 6, "100110101001": 6, "101000010010": 6,
    "101001001011": 6, "101011011010": 6, "101100001101": 6, "101101000011": 6, "101101000110": 6, "101110000110": 6,
    "101110000111": 6, "110000011010": 6, "110000111010": 6, "110000111100": 6, "110100100101": 6, "110110001011": 6,
    "110110011010": 6, "110110101001": 6, "000110101101": 5, "000111100010": 5, "001001100101": 5, "001001101001": 5,
    "001010010100": 5, "001010010110": 5, "001011010100": 5, "001101101101": 5, "001110011100": 5, "001111000011": 5,
    "001111010110": 5, "001111011010": 5, "010001111010": 5, "010011010010": 5, "010100101101": 5, "010101011010": 5,
    "010101101100": 5, "010110001010": 5, "010110001011": 5, "010110010010": 5, "010110100010": 5, "010110100110": 5,
    "010110101100": 5, "010110111010": 5, "010111101001": 5, "011000011110": 5, "011001011011": 5, "011001011110": 5,
    "011001101001": 5, "011010010100": 5, "011010100101": 5, "011011000111": 5, "011110001010": 5, "011111100101": 5,
    "100000111010": 5, "100100101100": 5, "100101010110": 5, "100101011010": 5, "100101110100": 5, "100110100110": 5,
    "100111101101": 5, "101000011000": 5, "101001010010": 5, "101001100110": 5, "101010100001": 5, "101011100101": 5,
    "101100101010": 5, "101101001011": 5, "101110000101": 5, "101110010010": 5, "101110010110": 5, "110010110101": 5,
    "110100001011": 5, "110100011110": 5, "110101010010": 5, "110110100011": 5, "111000011010": 5, "000111100101": 4,
    "001000011010": 4, "001001010010": 4,
}
USE_CACHED = False      # change to True only if your job is stuck
if USE_CACHED:
    counts = dict(CACHED_QAOA)
    best = max(counts,
               key=lambda b: cut_value(b, edges))
    print("Loaded the cached result:", best, cut_value(best, edges))
elif "counts" in globals():
    print("Cache not loaded. The cells below use the counts of the last run that finished, normally the cell above.")
else:
    print("No result yet. Run the lab cell above, or set USE_CACHED = True if your job is stuck.")

The second option sends the same QAOA circuit to another quantum computer. `isa` was rewritten for one IBM processor, so it does not fit a different machine. The cell below therefore sends `qc`, the circuit before transpiling, with the two angles filled in.

How to use the cell:

1. In the Setup section set `USE_HARDWARE = True` and `HARDWARE = "qbraid"`, then run the first two setup cells again. The first time, the setup cell installs the `qbraid` package and asks for a restart.
2. Run the cell below. It prints one line for each device, with the device's id, its status and its number of qubits. Then it stops with an error saying that the device `PASTE_DEVICE_ID` was not found. That stop is expected on the first run, because no device is chosen yet. Copy the id of a device that is online and has at least 12 qubits into `QBRAID_DEVICE` in the Setup section, run that cell, then run the cell below again.
3. The cell waits for the job. Then it scores the bitstrings as the lab did, and prints the best one with its cut value.

Four lines in the cell, from `formats = ...` to `set_target_program_type`, ask the device for the newer of two text formats in which a circuit can travel, OpenQASM 3. On the tested versions the `qbraid` package can only load that one.

Two things to know about the waiting. `job.wait_for_final_state()` also returns when a job has failed, and reading the counts then stops with `ValueError: Counts data is not available.` A job that is put on hold (status `HOLD`) never ends the wait. In that case interrupt the kernel and ask an organizer.

The cell keeps only the results that have 12 characters, one for each node, and it assumes that the device, like Qiskit, writes qubit 0 as the last character. The best cut cannot confirm that order: on this graph a result read from the wrong end still scores 15 in most runs. If you are not sure which end of a result holds qubit 0, look it up in the device's documentation or ask an organizer.

This cell was checked against the `qbraid` 0.13.0 package with the network replaced by a stand-in. It has not been run with a real qBraid account.

**Hardware cell, second option (qBraid).** With `USE_HARDWARE = True` and `HARDWARE = "qbraid"` this cell uses your qBraid account. With any other setting it is skipped and uses no account.

In [ ]:
provider = connect("qbraid")
if provider:
    for dev in provider.get_devices():
        print(dev.id, dev.status(), "| qubits:", dev.num_qubits)

    device = provider.get_device(QBRAID_DEVICE)
    formats = device.profile.program_spec   # the text formats this device accepts
    formats = formats if isinstance(formats, list) else [formats]
    if "qasm3" in [item.alias for item in formats]:
        device.set_target_program_type("qasm3")
    job = device.run(qc.assign_parameters(angles), shots=4096)    # the circuit before transpiling
    job.wait_for_final_state()
    device_counts = job.result().data.get_counts()
    full = {bits: shots for bits, shots in device_counts.items() if len(bits) == n}   # one character per node
    if full:
        counts = full
        best = max(counts,
                   key=lambda b: cut_value(b, edges))
        print(best, cut_value(best, edges))
    else:
        print("No result has", n, "characters. Does the device have at least", n, "qubits? It returned:", device_counts)

Is 15 the best possible cut? The graph is small enough to check every one of the 4,096 possible bitstrings. The next cell does that with the same `cut_value` function. It takes a few thousandths of a second.

In [ ]:
best_possible, winners = 0, []
for k in range(2 ** n):
    bits = format(k, "012b")                 # the number k written as 12 binary digits
    value = cut_value(bits, edges)
    if value > best_possible:
        best_possible, winners = value, [bits]
    elif value == best_possible:
        winners.append(bits)
print("best possible cut:", best_possible, "| reached by:", winners)
print("your best sample: ", best, "with a cut of", cut_value(best, edges))

The best possible cut is 15, and exactly two bitstrings reach it, the two mirror images from above. The graph has 16 edges, but a triangle always keeps at least one of its edges uncut, so 16 is out of reach. If your best sample has a cut of 15, your samples contain the best answer there is.

How much of that is due to the circuit? For a graph this small, luck would often be enough: 4,096 random guesses contain one of the two best bitstrings about 86 times in 100. What the circuit changes is how often good cuts come up. The next cell shows it. It sorts your shots by their cut value and compares them with random guessing, where each of the 4,096 bitstrings is equally likely.

In [ ]:
import matplotlib.pyplot as plt

total = sum(counts.values())
sampled = [0] * 17                           # sampled[v]: the number of your shots with the cut value v
for bits, shots in counts.items():
    sampled[cut_value(bits, edges)] += shots
guessing = [0] * 17                          # guessing[v]: how many of the 4,096 bitstrings have the cut value v
for k in range(2 ** n):
    guessing[cut_value(format(k, "012b"), edges)] += 1

print("shots:", total, "| different bitstrings among them:", len(counts))
print("shots with a cut of 15:", sampled[15], "| random guessing would give about", round(total * guessing[15] / 4096, 1))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([v - 0.2 for v in range(17)], [g / 4096 for g in guessing], width=0.4, color="0.6", label="random guessing")
ax.bar([v + 0.2 for v in range(17)], [s / total for s in sampled], width=0.4, color="tab:blue", label="your samples")
ax.set_xticks(range(0, 17, 2))
ax.set_xlabel("cut value")
ax.set_ylabel("fraction of the shots")
ax.set_title("Your samples against random guessing")
ax.legend()
plt.show()

The gray bars are random guessing. They are centered on a cut of 8, half of the 16 edges, and only 2 of the 4,096 bitstrings have a cut of 15. The blue bars are your samples. They sit further to the right, at larger cuts.

Some numbers to compare with. An ideal QAOA circuit with these angles, one without noise, gives an average cut of 10.94 and one of the two best bitstrings in about 2.8 percent of the shots, which is about 115 of 4,096. Random guessing gives an average of 8 and a best bitstring in about 2 of 4,096 shots. Noise pulls the samples back toward the gray bars. On the simulated processor, test runs gave about 80 shots with a cut of 15 on average, and anything between 60 and 105 is ordinary. They gave about 1,500 different bitstrings and an average cut near 10.6. The recorded run on the real processor ibm_fez gave 64 shots with a cut of 15, 1,566 different bitstrings and an average cut of 10.49.

This is how the lab reads a good answer out of noisy samples. The quantum circuit does not print the answer. It makes good bitstrings more likely, the processor adds errors, and a classical computer scores every sample and keeps the best. Scoring is cheap, 16 comparisons for each bitstring, so noise costs you a few good samples and not the answer.

The last cell of the lab draws your best sample as a cut of the graph.

In [ ]:
best_sides = [int(bit) for bit in best[::-1]]          # reversed, so that best_sides[i] is the bit of node i
draw_graph(best_sides, "Your best sample: " + str(cut_value(best, edges)) + " of 16 edges cross")

With a cut of 15 exactly one edge is drawn as a dotted red line: the edge between nodes 6 and 7, which belongs to both triangles.

#### Your turn 3

Complete the function `average_cut`. It takes a dictionary of counts and a list of edges, and returns the average cut value over all shots: add up the cut values of all the shots, then divide by the number of shots. A bitstring that was measured five times counts five times. Use `cut_value` from the lab for the cut value of one bitstring.

In [ ]:
def average_cut(counts, edges):
    """Return the average cut value over all the shots in counts."""
    mean = None
    # YOUR CODE HERE
    return mean

In [ ]:
from numbers import Integral, Real

first = average_cut({"000": 4}, [(0, 1), (1, 2)])
assert first is not None, "average_cut() still returns None. Replace the line YOUR CODE HERE with code that sets mean."
assert isinstance(first, Real), f"average_cut() should return one number, the average. Yours returned a {type(first).__name__}."
weighted = average_cut({"010": 3, "000": 1}, [(0, 1), (1, 2)])
assert not isinstance(weighted, Integral), "The average came back as a whole number. Divide the total cut by the total number of shots with /, not //, and do not round the result."
assert abs(weighted - 1.5) < 1e-9, "Weight each cut value by its number of shots, and divide by the total number of shots."
ordered = average_cut({"001": 1, "100": 1, "110": 2}, [(0, 1)])
assert abs(ordered - 0.75) < 1e-9, "Use cut_value() for every bitstring, so that qubit 0 is read from the right. Return the average as it is, without rounding it."
third = average_cut({"01": 1, "00": 2}, [(0, 1)])
assert abs(third - 1 / 3) < 1e-9, "Return the average as it is, without rounding it."
print("Check passed.")
print("average cut of your samples:   ", round(average_cut(counts, edges), 2))
print("average cut of random guessing:", len(edges) / 2)

## What you should see

*Slide 27.*

| Lab | Expected result |
|---|---|
| Lab 1: Grover on two qubits | 11 on every shot of the simulator |
| Lab 2: the QAOA circuit | One ZZ rotation per edge before transpiling, more two-qubit gates after |
| Lab 3: hardware samples | Many different bitstrings; the best one is your answer |

In numbers, with `USE_HARDWARE = False`: Lab 1 prints `{'11': 1024}`. Lab 2 has 16 `rzz` gates before transpiling and 53 `cz` gates after. Lab 3 returns about 1,500 different bitstrings in 4,096 shots, and its best sample has a cut of 15. The "Try it" cell prints `-1.8573`.

## If something breaks

*Slide 28.*

| What you see | What to do |
|---|---|
| The job stays queued | Load the cached result in the next cell and keep going |
| assign_parameters raises an error | One layer takes exactly two angles, in the order of the circuit's parameters |
| Cut values look wrong | Reverse the bitstring: qubit 0 is its last character |
| qaoa_ansatz cannot be imported | Use the Qiskit version the notebook pins |
| IBM rejects your key (an error that mentions the API key, the token or 401) | Open the Session 1 notebook, set `REPLACE_SAVED_KEY = True` in the cell under "Save your IBM credentials once", run it with a new key, then come back here |
| `NameError` for `edges`, `pm`, `isa` or `angles` | Run the cells of Lab 2 and Lab 3 in order, starting with the cell that defines `edges` and `pm` |
| The average cut is below 8 and no error appears | The two angles are swapped. The order is `[beta, gamma]` |

The last three rows are not on the slide. They were added after test runs and a review of this notebook.

How three of the rows look in this notebook. The error in the second row starts with `ValueError: Mismatching number of values and parameters.` on the tested versions. In the third row, `cut_value` already reverses the bitstring, so use that function whenever you score a bitstring yourself. The order matters for tonight's graph: read from the left, without reversing, each of the two best bitstrings scores 11 and not 15. For the fourth row, run the setup cell again: it installs the tested version of Qiskit and then asks for a restart.

About the last row: with the angles the wrong way round, the samples are worse than random guessing, with an average cut below 7 in a test run. The best single sample can still have a cut of 15 by luck, so look at the average and not at the best sample to notice this mistake.

## Check yourself

*Slide 23.*

1. How many Grover steps for $N = 16$ and $M = 1$?
2. What is the cut value when every node is on the same side?
3. Why can lowering $E(\theta)$ never go below the true minimum?

## Tonight in one slide

*Slide 29.*

- **Interference.** Mark the good answers, then make them add up.
- **Grover.** About $\sqrt{N}$ steps in place of $N/2$.
- **QAOA.** A cost function turned into a circuit; good angles raise the average cut.
- **VQE.** The same loop, with energy as the cost.

## Challenge 2: Max-Cut with QAOA

*Slide 30.*

Find the best cut of a 12-node graph from samples taken on real hardware.

Your score comes from the average cut of all 4,096 shots; the best cut must be among them.

Start from tonight's notebook. Submission steps are in the challenge notebook.

The challenge notebook is posted on the event page when the challenge opens, under Challenges, "Max-Cut with QAOA".

## Where to go from here

*Slides 31 and 32.*

- **Tonight.** Score your samples and start Challenge 2.
- **Try it.** Grover on three qubits: how many steps does it need?
- **Stuck?** Ask in Discord between sessions.

For the second card: three qubits make $N = 8$ items. Predict the number of steps with the formula from Part 1, then test your prediction. Copy the cell in Part 1 that runs Grover's search on four qubits, under "Each Grover step rotates the state by $2\theta$", and change it to three qubits. `QuantumCircuit(4)` becomes `QuantumCircuit(3)`, every `range(4)` becomes `range(3)`, `mcp(np.pi, [0, 1, 2], 3)` becomes `mcp(np.pi, [0, 1], 2)`, the bitstrings `wanted` and `"0000"` get three characters, and `np.arcsin(1 / 4)` becomes `np.arcsin(1 / np.sqrt(8))`.

The last slide is titled "Four algorithms, one pattern". The pattern is the title of tonight's first idea: interference, arranged on purpose.

- Slides and Challenge 2 are posted tonight.
- Challenges 1 and 2 both stay open until Sunday.
- Tomorrow at 5:00 PM: Session 3, quantum cryptography.